# BEH 30: rolling behaviour windows

The behavioural readings for every customer and month-end (`beh.CustomerMonthBehaviour`): each activity measure
summed over the 3, 12 and 24 calendar months ending at the month-end (3 months stands in for the brief's 90 days),
the opportunity counts beside them (months held, renewals due, messages sent), per-month rates and a recent (last 3
months) against prior (months 4 to 24) comparison.

Rules (clarifications B1 to B4): only activity on or before the month-end is used. A window that starts before its
source's coverage start is null, not zero (settings `cov_*`; online quotes only reach customers from 2026-05-06,
because of the GDPR wipe). A rate whose opportunity is zero is null. Zero means the customer had the opportunity
and did not act.

The windows are range frames over a customer x month grid (`stg.beh_grid`), so no 24-month self-join is needed. The
same pass computes the forward sums used for cost to serve in BEH_50.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain and key.
2. Fix the cause (usually a renamed or missing source column, or a broken key), then rerun from that cell.
   Every cell is `create or replace`, so rerunning is safe.
3. The checks cell at the end names any table that is empty or has duplicate / null keys, and the likely cause.

Run order: BEH_00_Setup is run by every notebook. Then 10, 20, 30, 40, 50, 60, in that order.


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run BEH_00_Setup

In [ ]:
%%pyspark
beh_run_log("BEH_30_Rolling", "started")


## beh_grid


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_grid                                                                      */
/*  Description : Customer x calendar month grid for customers in beh.CustomerMonth: activity,      */
/*                comms, months held and offered renewals due, one column per measure               */
/*                (build/measures.py).                                                              */
/*  Reads       : stg.beh_activity_month, stg.beh_comms_month, beh.CustomerMonth,                   */
/*                stg.beh_renewal_due                                                               */
/*  Key         : SCV_Customer_Key + MonthNo                                                        */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_grid as
select  u.SCV_Customer_Key,
        u.MonthNo,
        sum(u.prompts) as prompts,
        sum(u.ren_cust_actions) as ren_cust_actions,
        sum(u.inlife_contacts) as inlife_contacts,
        sum(u.friction_events) as friction_events,
        sum(u.friction_events_il) as friction_events_il,
        sum(u.doc_friction_events) as doc_friction_events,
        sum(u.chases) as chases,
        sum(u.chases_il) as chases_il,
        sum(u.concessions) as concessions,
        sum(u.failed_payments) as failed_payments,
        sum(u.failed_payments_il) as failed_payments_il,
        sum(u.failed_logins) as failed_logins,
        sum(u.arrears_events) as arrears_events,
        sum(u.doc_requests) as doc_requests,
        sum(u.sales_calls) as sales_calls,
        sum(u.sales_calls_il) as sales_calls_il,
        sum(u.events_total) as events_total,
        sum(u.online_quotes_motor) as online_quotes_motor,
        sum(u.online_quotes_motor_il) as online_quotes_motor_il,
        sum(u.online_quotes_home) as online_quotes_home,
        sum(u.online_quotes_home_il) as online_quotes_home_il,
        sum(u.online_quotes_van) as online_quotes_van,
        sum(u.online_quotes_van_il) as online_quotes_van_il,
        sum(u.renewal_requotes) as renewal_requotes,
        sum(u.episodes) as episodes,
        sum(u.episodes_il) as episodes_il,
        sum(u.episodes_friction) as episodes_friction,
        sum(u.episodes_friction_il) as episodes_friction_il,
        sum(u.episodes_completed) as episodes_completed,
        sum(u.calls_in) as calls_in,
        sum(u.calls_in_il) as calls_in_il,
        sum(u.calls_in_answered) as calls_in_answered,
        sum(u.calls_in_abandoned) as calls_in_abandoned,
        sum(u.calls_in_long_wait) as calls_in_long_wait,
        sum(u.calls_in_wait_s) as calls_in_wait_s,
        sum(u.calls_in_talk_s) as calls_in_talk_s,
        sum(u.calls_out) as calls_out,
        sum(u.calls_out_il) as calls_out_il,
        sum(u.calls_out_talk_s) as calls_out_talk_s,
        sum(u.mtas) as mtas,
        sum(u.mtas_il) as mtas_il,
        sum(u.mta_records) as mta_records,
        sum(u.mtas_forced) as mtas_forced,
        sum(u.mtas_elective) as mtas_elective,
        sum(u.mta_cat_cover) as mta_cat_cover,
        sum(u.mta_cat_drivers) as mta_cat_drivers,
        sum(u.mta_cat_vehicle) as mta_cat_vehicle,
        sum(u.mta_cat_address) as mta_cat_address,
        sum(u.mta_cat_licence) as mta_cat_licence,
        sum(u.mta_cat_use) as mta_cat_use,
        sum(u.mta_cat_personal) as mta_cat_personal,
        sum(u.loan_status_changes) as loan_status_changes,
        sum(u.msgs_sent) as msgs_sent,
        sum(u.msgs_sent_il) as msgs_sent_il,
        sum(u.msgs_opened) as msgs_opened,
        sum(u.msgs_opened_il) as msgs_opened_il,
        sum(u.msgs_clicked) as msgs_clicked,
        sum(u.msgs_clicked_il) as msgs_clicked_il,
        sum(u.msgs_bounced) as msgs_bounced,
        sum(u.episodes_assisted) as episodes_assisted,
        sum(u.episodes_channel_tagged) as episodes_channel_tagged,
        sum(u.channel_switches) as channel_switches,
        sum(u.episodes_repeat_contact) as episodes_repeat_contact,
        sum(u.nb_sales) as nb_sales,
        sum(u.nb_sales_il) as nb_sales_il,
        sum(u.nb_sales_web_assist) as nb_sales_web_assist,
        sum(u.nb_sales_web_assist_il) as nb_sales_web_assist_il,
        sum(u.doc_decisions) as doc_decisions,
        sum(u.doc_rejections) as doc_rejections,
        sum(u.doc_attempts) as doc_attempts,
        sum(u.doc_nonreceipt) as doc_nonreceipt,
        sum(u.doc_nonreceipt_il) as doc_nonreceipt_il,
        sum(u.doc_nonreceipt_premium) as doc_nonreceipt_premium,
        sum(u.consent_actions) as consent_actions,
        sum(u.consent_actions_il) as consent_actions_il,
        sum(u.consent_withdrawals) as consent_withdrawals,
        sum(u.consent_withdrawals_il) as consent_withdrawals_il,
        sum(u.mfq_quotes) as mfq_quotes,
        sum(u.mfq_retrievals) as mfq_retrievals,
        sum(u.mfq_insurers) as mfq_insurers,
        sum(u.mfq_returned) as mfq_returned,
        sum(u.mfq_cover_compared) as mfq_cover_compared,
        sum(u.mfq_cover_down) as mfq_cover_down,
        sum(u.mfq_cover_up) as mfq_cover_up,
        sum(u.mfq_device_tagged) as mfq_device_tagged,
        sum(u.mfq_device_switch) as mfq_device_switch,
        sum(u.web_sessions) as web_sessions,
        sum(u.web_sessions_nonquote) as web_sessions_nonquote,
        sum(u.web_page_views) as web_page_views,
        sum(u.web_content_views) as web_content_views,
        sum(u.web_other_product_sessions) as web_other_product_sessions,
        sum(u.calls_other_queues) as calls_other_queues,
        sum(u.friction_all) as friction_all,
        sum(u.friction_resolved) as friction_resolved,
        sum(u.friction_payment) as friction_payment,
        sum(u.friction_payment_resolved) as friction_payment_resolved,
        sum(u.friction_login) as friction_login,
        sum(u.friction_login_resolved) as friction_login_resolved,
        sum(u.friction_contact) as friction_contact,
        sum(u.friction_contact_resolved) as friction_contact_resolved,
        sum(u.friction_documents) as friction_documents,
        sum(u.friction_documents_resolved) as friction_documents_resolved,
        sum(u.friction_quote) as friction_quote,
        sum(u.friction_quote_resolved) as friction_quote_resolved,
        sum(u.friction_process) as friction_process,
        sum(u.friction_process_resolved) as friction_process_resolved,
        sum(u.friction_other) as friction_other,
        sum(u.friction_other_resolved) as friction_other_resolved,
        sum(u.offers_sent) as offers_sent,
        sum(u.offers_clicked) as offers_clicked,
        sum(u.offer_clicks_nb) as offer_clicks_nb,
        sum(u.offer_clicks_renewal) as offer_clicks_renewal,
        sum(u.friction_v3_eligible) as friction_v3_eligible,
        sum(u.friction_v3_affected) as friction_v3_affected,
        sum(u.friction_v3_res_known) as friction_v3_res_known,
        sum(u.friction_v3_resolved) as friction_v3_resolved,
        sum(u.friction_v3_renewal_eligible) as friction_v3_renewal_eligible,
        sum(u.friction_v3_renewal_affected) as friction_v3_renewal_affected,
        sum(u.friction_v3_renewal_res_known) as friction_v3_renewal_res_known,
        sum(u.friction_v3_renewal_resolved) as friction_v3_renewal_resolved,
        sum(u.friction_v3_firstyear_eligible) as friction_v3_firstyear_eligible,
        sum(u.friction_v3_firstyear_affected) as friction_v3_firstyear_affected,
        sum(u.friction_v3_firstyear_res_known) as friction_v3_firstyear_res_known,
        sum(u.friction_v3_firstyear_resolved) as friction_v3_firstyear_resolved,
        sum(u.friction_v3_change_eligible) as friction_v3_change_eligible,
        sum(u.friction_v3_change_affected) as friction_v3_change_affected,
        sum(u.friction_v3_change_res_known) as friction_v3_change_res_known,
        sum(u.friction_v3_change_resolved) as friction_v3_change_resolved,
        sum(u.friction_v3_docreq_eligible) as friction_v3_docreq_eligible,
        sum(u.friction_v3_docreq_affected) as friction_v3_docreq_affected,
        sum(u.friction_v3_docreq_res_known) as friction_v3_docreq_res_known,
        sum(u.friction_v3_docreq_resolved) as friction_v3_docreq_resolved,
        sum(u.months_held) as months_held,
        sum(u.renewals_due) as renewals_due,
        sum(u.motor_renewals_due) as motor_renewals_due
from    (select  a.SCV_Customer_Key, a.MonthNo,
                 a.prompts as prompts,
        a.ren_cust_actions as ren_cust_actions,
        a.inlife_contacts as inlife_contacts,
        a.friction_events as friction_events,
        a.friction_events_il as friction_events_il,
        a.doc_friction_events as doc_friction_events,
        a.chases as chases,
        a.chases_il as chases_il,
        a.concessions as concessions,
        a.failed_payments as failed_payments,
        a.failed_payments_il as failed_payments_il,
        a.failed_logins as failed_logins,
        a.arrears_events as arrears_events,
        a.doc_requests as doc_requests,
        a.sales_calls as sales_calls,
        a.sales_calls_il as sales_calls_il,
        a.events_total as events_total,
        a.online_quotes_motor as online_quotes_motor,
        a.online_quotes_motor_il as online_quotes_motor_il,
        a.online_quotes_home as online_quotes_home,
        a.online_quotes_home_il as online_quotes_home_il,
        a.online_quotes_van as online_quotes_van,
        a.online_quotes_van_il as online_quotes_van_il,
        a.renewal_requotes as renewal_requotes,
        a.episodes as episodes,
        a.episodes_il as episodes_il,
        a.episodes_friction as episodes_friction,
        a.episodes_friction_il as episodes_friction_il,
        a.episodes_completed as episodes_completed,
        a.calls_in as calls_in,
        a.calls_in_il as calls_in_il,
        a.calls_in_answered as calls_in_answered,
        a.calls_in_abandoned as calls_in_abandoned,
        a.calls_in_long_wait as calls_in_long_wait,
        a.calls_in_wait_s as calls_in_wait_s,
        a.calls_in_talk_s as calls_in_talk_s,
        a.calls_out as calls_out,
        a.calls_out_il as calls_out_il,
        a.calls_out_talk_s as calls_out_talk_s,
        a.mtas as mtas,
        a.mtas_il as mtas_il,
        a.mta_records as mta_records,
        a.mtas_forced as mtas_forced,
        a.mtas_elective as mtas_elective,
        a.mta_cat_cover as mta_cat_cover,
        a.mta_cat_drivers as mta_cat_drivers,
        a.mta_cat_vehicle as mta_cat_vehicle,
        a.mta_cat_address as mta_cat_address,
        a.mta_cat_licence as mta_cat_licence,
        a.mta_cat_use as mta_cat_use,
        a.mta_cat_personal as mta_cat_personal,
        a.loan_status_changes as loan_status_changes,
        0 as msgs_sent,
        0 as msgs_sent_il,
        0 as msgs_opened,
        0 as msgs_opened_il,
        0 as msgs_clicked,
        0 as msgs_clicked_il,
        0 as msgs_bounced,
        a.episodes_assisted as episodes_assisted,
        a.episodes_channel_tagged as episodes_channel_tagged,
        a.channel_switches as channel_switches,
        a.episodes_repeat_contact as episodes_repeat_contact,
        a.nb_sales as nb_sales,
        a.nb_sales_il as nb_sales_il,
        a.nb_sales_web_assist as nb_sales_web_assist,
        a.nb_sales_web_assist_il as nb_sales_web_assist_il,
        a.doc_decisions as doc_decisions,
        a.doc_rejections as doc_rejections,
        a.doc_attempts as doc_attempts,
        a.doc_nonreceipt as doc_nonreceipt,
        a.doc_nonreceipt_il as doc_nonreceipt_il,
        a.doc_nonreceipt_premium as doc_nonreceipt_premium,
        a.consent_actions as consent_actions,
        a.consent_actions_il as consent_actions_il,
        a.consent_withdrawals as consent_withdrawals,
        a.consent_withdrawals_il as consent_withdrawals_il,
        a.mfq_quotes as mfq_quotes,
        a.mfq_retrievals as mfq_retrievals,
        a.mfq_insurers as mfq_insurers,
        a.mfq_returned as mfq_returned,
        a.mfq_cover_compared as mfq_cover_compared,
        a.mfq_cover_down as mfq_cover_down,
        a.mfq_cover_up as mfq_cover_up,
        a.mfq_device_tagged as mfq_device_tagged,
        a.mfq_device_switch as mfq_device_switch,
        a.web_sessions as web_sessions,
        a.web_sessions_nonquote as web_sessions_nonquote,
        a.web_page_views as web_page_views,
        a.web_content_views as web_content_views,
        a.web_other_product_sessions as web_other_product_sessions,
        a.calls_other_queues as calls_other_queues,
        a.friction_all as friction_all,
        a.friction_resolved as friction_resolved,
        a.friction_payment as friction_payment,
        a.friction_payment_resolved as friction_payment_resolved,
        a.friction_login as friction_login,
        a.friction_login_resolved as friction_login_resolved,
        a.friction_contact as friction_contact,
        a.friction_contact_resolved as friction_contact_resolved,
        a.friction_documents as friction_documents,
        a.friction_documents_resolved as friction_documents_resolved,
        a.friction_quote as friction_quote,
        a.friction_quote_resolved as friction_quote_resolved,
        a.friction_process as friction_process,
        a.friction_process_resolved as friction_process_resolved,
        a.friction_other as friction_other,
        a.friction_other_resolved as friction_other_resolved,
        a.offers_sent as offers_sent,
        a.offers_clicked as offers_clicked,
        a.offer_clicks_nb as offer_clicks_nb,
        a.offer_clicks_renewal as offer_clicks_renewal,
        a.friction_v3_eligible as friction_v3_eligible,
        a.friction_v3_affected as friction_v3_affected,
        a.friction_v3_res_known as friction_v3_res_known,
        a.friction_v3_resolved as friction_v3_resolved,
        a.friction_v3_renewal_eligible as friction_v3_renewal_eligible,
        a.friction_v3_renewal_affected as friction_v3_renewal_affected,
        a.friction_v3_renewal_res_known as friction_v3_renewal_res_known,
        a.friction_v3_renewal_resolved as friction_v3_renewal_resolved,
        a.friction_v3_firstyear_eligible as friction_v3_firstyear_eligible,
        a.friction_v3_firstyear_affected as friction_v3_firstyear_affected,
        a.friction_v3_firstyear_res_known as friction_v3_firstyear_res_known,
        a.friction_v3_firstyear_resolved as friction_v3_firstyear_resolved,
        a.friction_v3_change_eligible as friction_v3_change_eligible,
        a.friction_v3_change_affected as friction_v3_change_affected,
        a.friction_v3_change_res_known as friction_v3_change_res_known,
        a.friction_v3_change_resolved as friction_v3_change_resolved,
        a.friction_v3_docreq_eligible as friction_v3_docreq_eligible,
        a.friction_v3_docreq_affected as friction_v3_docreq_affected,
        a.friction_v3_docreq_res_known as friction_v3_docreq_res_known,
        a.friction_v3_docreq_resolved as friction_v3_docreq_resolved,
        0 as months_held,
        0 as renewals_due,
        0 as motor_renewals_due
         from    ${beh.stg}.beh_activity_month a
         union all
         select  m.SCV_Customer_Key, m.MonthNo,
                 0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as friction_events_il,
        0 as doc_friction_events,
        0 as chases,
        0 as chases_il,
        0 as concessions,
        0 as failed_payments,
        0 as failed_payments_il,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as sales_calls_il,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_motor_il,
        0 as online_quotes_home,
        0 as online_quotes_home_il,
        0 as online_quotes_van,
        0 as online_quotes_van_il,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_il,
        0 as episodes_friction,
        0 as episodes_friction_il,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_il,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_il,
        0 as calls_out_talk_s,
        0 as mtas,
        0 as mtas_il,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        m.msgs_sent as msgs_sent,
        m.msgs_sent_il as msgs_sent_il,
        m.msgs_opened as msgs_opened,
        m.msgs_opened_il as msgs_opened_il,
        m.msgs_clicked as msgs_clicked,
        m.msgs_clicked_il as msgs_clicked_il,
        m.msgs_bounced as msgs_bounced,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_il,
        0 as nb_sales_web_assist,
        0 as nb_sales_web_assist_il,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_il,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_actions_il,
        0 as consent_withdrawals,
        0 as consent_withdrawals_il,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved,
        0 as months_held,
        0 as renewals_due,
        0 as motor_renewals_due
         from    ${beh.stg}.beh_comms_month m
         union all
         select  c.SCV_Customer_Key, c.MonthNo,
                 0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as friction_events_il,
        0 as doc_friction_events,
        0 as chases,
        0 as chases_il,
        0 as concessions,
        0 as failed_payments,
        0 as failed_payments_il,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as sales_calls_il,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_motor_il,
        0 as online_quotes_home,
        0 as online_quotes_home_il,
        0 as online_quotes_van,
        0 as online_quotes_van_il,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_il,
        0 as episodes_friction,
        0 as episodes_friction_il,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_il,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_il,
        0 as calls_out_talk_s,
        0 as mtas,
        0 as mtas_il,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        0 as msgs_sent,
        0 as msgs_sent_il,
        0 as msgs_opened,
        0 as msgs_opened_il,
        0 as msgs_clicked,
        0 as msgs_clicked_il,
        0 as msgs_bounced,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_il,
        0 as nb_sales_web_assist,
        0 as nb_sales_web_assist_il,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_il,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_actions_il,
        0 as consent_withdrawals,
        0 as consent_withdrawals_il,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved,
        1 as months_held,
        0 as renewals_due,
        0 as motor_renewals_due
         from    ${beh.target}.CustomerMonth c
         union all
         select  d.SCV_Customer_Key, d.MonthNo,
                 0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as friction_events_il,
        0 as doc_friction_events,
        0 as chases,
        0 as chases_il,
        0 as concessions,
        0 as failed_payments,
        0 as failed_payments_il,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as sales_calls_il,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_motor_il,
        0 as online_quotes_home,
        0 as online_quotes_home_il,
        0 as online_quotes_van,
        0 as online_quotes_van_il,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_il,
        0 as episodes_friction,
        0 as episodes_friction_il,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_il,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_il,
        0 as calls_out_talk_s,
        0 as mtas,
        0 as mtas_il,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        0 as msgs_sent,
        0 as msgs_sent_il,
        0 as msgs_opened,
        0 as msgs_opened_il,
        0 as msgs_clicked,
        0 as msgs_clicked_il,
        0 as msgs_bounced,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_il,
        0 as nb_sales_web_assist,
        0 as nb_sales_web_assist_il,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_il,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_actions_il,
        0 as consent_withdrawals,
        0 as consent_withdrawals_il,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved,
        0 as months_held,
        1 as renewals_due,
        case when d.PolicyTypeGroup = 'Motor' then 1 else 0 end as motor_renewals_due
         from    ${beh.stg}.beh_renewal_due d
         where   d.PolicyOfferNum = 1) u
join    (select distinct SCV_Customer_Key from ${beh.target}.CustomerMonth) k on k.SCV_Customer_Key = u.SCV_Customer_Key
group by u.SCV_Customer_Key, u.MonthNo


## beh_rolling


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_rolling                                                                   */
/*  Description : Raw window sums per customer month-end: each measure over the last 3, 12 and 24   */
/*                months, and the cost-to-serve measures over the next 3 and 12 months. No          */
/*                coverage rule applied yet.                                                        */
/*  Reads       : stg.beh_grid, beh.CustomerMonth                                                   */
/*  Key         : SCV_Customer_Key + MonthEndDate                                                   */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_rolling as
select  c.SCV_Customer_Key,
        c.MonthEndDate,
        c.MonthNo,
        w.prompts_3m,
        w.prompts_12m,
        w.prompts_24m,
        w.ren_cust_actions_3m,
        w.ren_cust_actions_12m,
        w.ren_cust_actions_24m,
        w.inlife_contacts_3m,
        w.inlife_contacts_12m,
        w.inlife_contacts_24m,
        w.friction_events_3m,
        w.friction_events_12m,
        w.friction_events_24m,
        w.friction_events_il_3m,
        w.friction_events_il_12m,
        w.friction_events_il_24m,
        w.doc_friction_events_3m,
        w.doc_friction_events_12m,
        w.doc_friction_events_24m,
        w.chases_3m,
        w.chases_12m,
        w.chases_24m,
        w.chases_il_3m,
        w.chases_il_12m,
        w.chases_il_24m,
        w.concessions_3m,
        w.concessions_12m,
        w.concessions_24m,
        w.failed_payments_3m,
        w.failed_payments_12m,
        w.failed_payments_24m,
        w.failed_payments_il_3m,
        w.failed_payments_il_12m,
        w.failed_payments_il_24m,
        w.failed_logins_3m,
        w.failed_logins_12m,
        w.failed_logins_24m,
        w.arrears_events_3m,
        w.arrears_events_12m,
        w.arrears_events_24m,
        w.doc_requests_3m,
        w.doc_requests_12m,
        w.doc_requests_24m,
        w.sales_calls_3m,
        w.sales_calls_12m,
        w.sales_calls_24m,
        w.sales_calls_il_3m,
        w.sales_calls_il_12m,
        w.sales_calls_il_24m,
        w.events_total_3m,
        w.events_total_12m,
        w.events_total_24m,
        w.online_quotes_motor_3m,
        w.online_quotes_motor_12m,
        w.online_quotes_motor_24m,
        w.online_quotes_motor_il_3m,
        w.online_quotes_motor_il_12m,
        w.online_quotes_motor_il_24m,
        w.online_quotes_home_3m,
        w.online_quotes_home_12m,
        w.online_quotes_home_24m,
        w.online_quotes_home_il_3m,
        w.online_quotes_home_il_12m,
        w.online_quotes_home_il_24m,
        w.online_quotes_van_3m,
        w.online_quotes_van_12m,
        w.online_quotes_van_24m,
        w.online_quotes_van_il_3m,
        w.online_quotes_van_il_12m,
        w.online_quotes_van_il_24m,
        w.renewal_requotes_3m,
        w.renewal_requotes_12m,
        w.renewal_requotes_24m,
        w.episodes_3m,
        w.episodes_12m,
        w.episodes_24m,
        w.episodes_il_3m,
        w.episodes_il_12m,
        w.episodes_il_24m,
        w.episodes_friction_3m,
        w.episodes_friction_12m,
        w.episodes_friction_24m,
        w.episodes_friction_il_3m,
        w.episodes_friction_il_12m,
        w.episodes_friction_il_24m,
        w.episodes_completed_3m,
        w.episodes_completed_12m,
        w.episodes_completed_24m,
        w.calls_in_3m,
        w.calls_in_12m,
        w.calls_in_24m,
        w.calls_in_il_3m,
        w.calls_in_il_12m,
        w.calls_in_il_24m,
        w.calls_in_answered_3m,
        w.calls_in_answered_12m,
        w.calls_in_answered_24m,
        w.calls_in_abandoned_3m,
        w.calls_in_abandoned_12m,
        w.calls_in_abandoned_24m,
        w.calls_in_long_wait_3m,
        w.calls_in_long_wait_12m,
        w.calls_in_long_wait_24m,
        w.calls_in_wait_s_3m,
        w.calls_in_wait_s_12m,
        w.calls_in_wait_s_24m,
        w.calls_in_talk_s_3m,
        w.calls_in_talk_s_12m,
        w.calls_in_talk_s_24m,
        w.calls_out_3m,
        w.calls_out_12m,
        w.calls_out_24m,
        w.calls_out_il_3m,
        w.calls_out_il_12m,
        w.calls_out_il_24m,
        w.calls_out_talk_s_3m,
        w.calls_out_talk_s_12m,
        w.calls_out_talk_s_24m,
        w.mtas_3m,
        w.mtas_12m,
        w.mtas_24m,
        w.mtas_il_3m,
        w.mtas_il_12m,
        w.mtas_il_24m,
        w.mta_records_3m,
        w.mta_records_12m,
        w.mta_records_24m,
        w.mtas_forced_3m,
        w.mtas_forced_12m,
        w.mtas_forced_24m,
        w.mtas_elective_3m,
        w.mtas_elective_12m,
        w.mtas_elective_24m,
        w.mta_cat_cover_3m,
        w.mta_cat_cover_12m,
        w.mta_cat_cover_24m,
        w.mta_cat_drivers_3m,
        w.mta_cat_drivers_12m,
        w.mta_cat_drivers_24m,
        w.mta_cat_vehicle_3m,
        w.mta_cat_vehicle_12m,
        w.mta_cat_vehicle_24m,
        w.mta_cat_address_3m,
        w.mta_cat_address_12m,
        w.mta_cat_address_24m,
        w.mta_cat_licence_3m,
        w.mta_cat_licence_12m,
        w.mta_cat_licence_24m,
        w.mta_cat_use_3m,
        w.mta_cat_use_12m,
        w.mta_cat_use_24m,
        w.mta_cat_personal_3m,
        w.mta_cat_personal_12m,
        w.mta_cat_personal_24m,
        w.loan_status_changes_3m,
        w.loan_status_changes_12m,
        w.loan_status_changes_24m,
        w.msgs_sent_3m,
        w.msgs_sent_12m,
        w.msgs_sent_24m,
        w.msgs_sent_il_3m,
        w.msgs_sent_il_12m,
        w.msgs_sent_il_24m,
        w.msgs_opened_3m,
        w.msgs_opened_12m,
        w.msgs_opened_24m,
        w.msgs_opened_il_3m,
        w.msgs_opened_il_12m,
        w.msgs_opened_il_24m,
        w.msgs_clicked_3m,
        w.msgs_clicked_12m,
        w.msgs_clicked_24m,
        w.msgs_clicked_il_3m,
        w.msgs_clicked_il_12m,
        w.msgs_clicked_il_24m,
        w.msgs_bounced_3m,
        w.msgs_bounced_12m,
        w.msgs_bounced_24m,
        w.episodes_assisted_3m,
        w.episodes_assisted_12m,
        w.episodes_assisted_24m,
        w.episodes_channel_tagged_3m,
        w.episodes_channel_tagged_12m,
        w.episodes_channel_tagged_24m,
        w.channel_switches_3m,
        w.channel_switches_12m,
        w.channel_switches_24m,
        w.episodes_repeat_contact_3m,
        w.episodes_repeat_contact_12m,
        w.episodes_repeat_contact_24m,
        w.nb_sales_3m,
        w.nb_sales_12m,
        w.nb_sales_24m,
        w.nb_sales_il_3m,
        w.nb_sales_il_12m,
        w.nb_sales_il_24m,
        w.nb_sales_web_assist_3m,
        w.nb_sales_web_assist_12m,
        w.nb_sales_web_assist_24m,
        w.nb_sales_web_assist_il_3m,
        w.nb_sales_web_assist_il_12m,
        w.nb_sales_web_assist_il_24m,
        w.doc_decisions_3m,
        w.doc_decisions_12m,
        w.doc_decisions_24m,
        w.doc_rejections_3m,
        w.doc_rejections_12m,
        w.doc_rejections_24m,
        w.doc_attempts_3m,
        w.doc_attempts_12m,
        w.doc_attempts_24m,
        w.doc_nonreceipt_3m,
        w.doc_nonreceipt_12m,
        w.doc_nonreceipt_24m,
        w.doc_nonreceipt_il_3m,
        w.doc_nonreceipt_il_12m,
        w.doc_nonreceipt_il_24m,
        w.doc_nonreceipt_premium_3m,
        w.doc_nonreceipt_premium_12m,
        w.doc_nonreceipt_premium_24m,
        w.consent_actions_3m,
        w.consent_actions_12m,
        w.consent_actions_24m,
        w.consent_actions_il_3m,
        w.consent_actions_il_12m,
        w.consent_actions_il_24m,
        w.consent_withdrawals_3m,
        w.consent_withdrawals_12m,
        w.consent_withdrawals_24m,
        w.consent_withdrawals_il_3m,
        w.consent_withdrawals_il_12m,
        w.consent_withdrawals_il_24m,
        w.mfq_quotes_3m,
        w.mfq_quotes_12m,
        w.mfq_quotes_24m,
        w.mfq_retrievals_3m,
        w.mfq_retrievals_12m,
        w.mfq_retrievals_24m,
        w.mfq_insurers_3m,
        w.mfq_insurers_12m,
        w.mfq_insurers_24m,
        w.mfq_returned_3m,
        w.mfq_returned_12m,
        w.mfq_returned_24m,
        w.mfq_cover_compared_3m,
        w.mfq_cover_compared_12m,
        w.mfq_cover_compared_24m,
        w.mfq_cover_down_3m,
        w.mfq_cover_down_12m,
        w.mfq_cover_down_24m,
        w.mfq_cover_up_3m,
        w.mfq_cover_up_12m,
        w.mfq_cover_up_24m,
        w.mfq_device_tagged_3m,
        w.mfq_device_tagged_12m,
        w.mfq_device_tagged_24m,
        w.mfq_device_switch_3m,
        w.mfq_device_switch_12m,
        w.mfq_device_switch_24m,
        w.web_sessions_3m,
        w.web_sessions_12m,
        w.web_sessions_24m,
        w.web_sessions_nonquote_3m,
        w.web_sessions_nonquote_12m,
        w.web_sessions_nonquote_24m,
        w.web_page_views_3m,
        w.web_page_views_12m,
        w.web_page_views_24m,
        w.web_content_views_3m,
        w.web_content_views_12m,
        w.web_content_views_24m,
        w.web_other_product_sessions_3m,
        w.web_other_product_sessions_12m,
        w.web_other_product_sessions_24m,
        w.calls_other_queues_3m,
        w.calls_other_queues_12m,
        w.calls_other_queues_24m,
        w.friction_all_3m,
        w.friction_all_12m,
        w.friction_all_24m,
        w.friction_resolved_3m,
        w.friction_resolved_12m,
        w.friction_resolved_24m,
        w.friction_payment_3m,
        w.friction_payment_12m,
        w.friction_payment_24m,
        w.friction_payment_resolved_3m,
        w.friction_payment_resolved_12m,
        w.friction_payment_resolved_24m,
        w.friction_login_3m,
        w.friction_login_12m,
        w.friction_login_24m,
        w.friction_login_resolved_3m,
        w.friction_login_resolved_12m,
        w.friction_login_resolved_24m,
        w.friction_contact_3m,
        w.friction_contact_12m,
        w.friction_contact_24m,
        w.friction_contact_resolved_3m,
        w.friction_contact_resolved_12m,
        w.friction_contact_resolved_24m,
        w.friction_documents_3m,
        w.friction_documents_12m,
        w.friction_documents_24m,
        w.friction_documents_resolved_3m,
        w.friction_documents_resolved_12m,
        w.friction_documents_resolved_24m,
        w.friction_quote_3m,
        w.friction_quote_12m,
        w.friction_quote_24m,
        w.friction_quote_resolved_3m,
        w.friction_quote_resolved_12m,
        w.friction_quote_resolved_24m,
        w.friction_process_3m,
        w.friction_process_12m,
        w.friction_process_24m,
        w.friction_process_resolved_3m,
        w.friction_process_resolved_12m,
        w.friction_process_resolved_24m,
        w.friction_other_3m,
        w.friction_other_12m,
        w.friction_other_24m,
        w.friction_other_resolved_3m,
        w.friction_other_resolved_12m,
        w.friction_other_resolved_24m,
        w.offers_sent_3m,
        w.offers_sent_12m,
        w.offers_sent_24m,
        w.offers_clicked_3m,
        w.offers_clicked_12m,
        w.offers_clicked_24m,
        w.offer_clicks_nb_3m,
        w.offer_clicks_nb_12m,
        w.offer_clicks_nb_24m,
        w.offer_clicks_renewal_3m,
        w.offer_clicks_renewal_12m,
        w.offer_clicks_renewal_24m,
        w.friction_v3_eligible_3m,
        w.friction_v3_eligible_12m,
        w.friction_v3_eligible_24m,
        w.friction_v3_affected_3m,
        w.friction_v3_affected_12m,
        w.friction_v3_affected_24m,
        w.friction_v3_res_known_3m,
        w.friction_v3_res_known_12m,
        w.friction_v3_res_known_24m,
        w.friction_v3_resolved_3m,
        w.friction_v3_resolved_12m,
        w.friction_v3_resolved_24m,
        w.friction_v3_renewal_eligible_3m,
        w.friction_v3_renewal_eligible_12m,
        w.friction_v3_renewal_eligible_24m,
        w.friction_v3_renewal_affected_3m,
        w.friction_v3_renewal_affected_12m,
        w.friction_v3_renewal_affected_24m,
        w.friction_v3_renewal_res_known_3m,
        w.friction_v3_renewal_res_known_12m,
        w.friction_v3_renewal_res_known_24m,
        w.friction_v3_renewal_resolved_3m,
        w.friction_v3_renewal_resolved_12m,
        w.friction_v3_renewal_resolved_24m,
        w.friction_v3_firstyear_eligible_3m,
        w.friction_v3_firstyear_eligible_12m,
        w.friction_v3_firstyear_eligible_24m,
        w.friction_v3_firstyear_affected_3m,
        w.friction_v3_firstyear_affected_12m,
        w.friction_v3_firstyear_affected_24m,
        w.friction_v3_firstyear_res_known_3m,
        w.friction_v3_firstyear_res_known_12m,
        w.friction_v3_firstyear_res_known_24m,
        w.friction_v3_firstyear_resolved_3m,
        w.friction_v3_firstyear_resolved_12m,
        w.friction_v3_firstyear_resolved_24m,
        w.friction_v3_change_eligible_3m,
        w.friction_v3_change_eligible_12m,
        w.friction_v3_change_eligible_24m,
        w.friction_v3_change_affected_3m,
        w.friction_v3_change_affected_12m,
        w.friction_v3_change_affected_24m,
        w.friction_v3_change_res_known_3m,
        w.friction_v3_change_res_known_12m,
        w.friction_v3_change_res_known_24m,
        w.friction_v3_change_resolved_3m,
        w.friction_v3_change_resolved_12m,
        w.friction_v3_change_resolved_24m,
        w.friction_v3_docreq_eligible_3m,
        w.friction_v3_docreq_eligible_12m,
        w.friction_v3_docreq_eligible_24m,
        w.friction_v3_docreq_affected_3m,
        w.friction_v3_docreq_affected_12m,
        w.friction_v3_docreq_affected_24m,
        w.friction_v3_docreq_res_known_3m,
        w.friction_v3_docreq_res_known_12m,
        w.friction_v3_docreq_res_known_24m,
        w.friction_v3_docreq_resolved_3m,
        w.friction_v3_docreq_resolved_12m,
        w.friction_v3_docreq_resolved_24m,
        w.months_held_3m,
        w.months_held_12m,
        w.months_held_24m,
        w.renewals_due_3m,
        w.renewals_due_12m,
        w.renewals_due_24m,
        w.motor_renewals_due_3m,
        w.motor_renewals_due_12m,
        w.motor_renewals_due_24m,
        w.calls_in_next3m,
        w.calls_in_next12m,
        w.calls_in_talk_s_next3m,
        w.calls_in_talk_s_next12m,
        w.calls_out_next3m,
        w.calls_out_next12m,
        w.calls_out_talk_s_next3m,
        w.calls_out_talk_s_next12m,
        w.inlife_contacts_next3m,
        w.inlife_contacts_next12m,
        w.mtas_next3m,
        w.mtas_next12m,
        w.doc_requests_next3m,
        w.doc_requests_next12m,
        w.chases_next3m,
        w.chases_next12m,
        w.failed_payments_next3m,
        w.failed_payments_next12m,
        w.friction_events_next3m,
        w.friction_events_next12m,
        w.arrears_events_next3m,
        w.arrears_events_next12m
from    (select  SCV_Customer_Key,
                 MonthNo,
                 sum(prompts) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as prompts_3m,
                 sum(prompts) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as prompts_12m,
                 sum(prompts) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as prompts_24m,
                 sum(ren_cust_actions) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as ren_cust_actions_3m,
                 sum(ren_cust_actions) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as ren_cust_actions_12m,
                 sum(ren_cust_actions) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as ren_cust_actions_24m,
                 sum(inlife_contacts) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as inlife_contacts_3m,
                 sum(inlife_contacts) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as inlife_contacts_12m,
                 sum(inlife_contacts) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as inlife_contacts_24m,
                 sum(friction_events) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_events_3m,
                 sum(friction_events) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_events_12m,
                 sum(friction_events) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_events_24m,
                 sum(friction_events_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_events_il_3m,
                 sum(friction_events_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_events_il_12m,
                 sum(friction_events_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_events_il_24m,
                 sum(doc_friction_events) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as doc_friction_events_3m,
                 sum(doc_friction_events) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as doc_friction_events_12m,
                 sum(doc_friction_events) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as doc_friction_events_24m,
                 sum(chases) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as chases_3m,
                 sum(chases) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as chases_12m,
                 sum(chases) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as chases_24m,
                 sum(chases_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as chases_il_3m,
                 sum(chases_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as chases_il_12m,
                 sum(chases_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as chases_il_24m,
                 sum(concessions) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as concessions_3m,
                 sum(concessions) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as concessions_12m,
                 sum(concessions) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as concessions_24m,
                 sum(failed_payments) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as failed_payments_3m,
                 sum(failed_payments) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as failed_payments_12m,
                 sum(failed_payments) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as failed_payments_24m,
                 sum(failed_payments_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as failed_payments_il_3m,
                 sum(failed_payments_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as failed_payments_il_12m,
                 sum(failed_payments_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as failed_payments_il_24m,
                 sum(failed_logins) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as failed_logins_3m,
                 sum(failed_logins) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as failed_logins_12m,
                 sum(failed_logins) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as failed_logins_24m,
                 sum(arrears_events) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as arrears_events_3m,
                 sum(arrears_events) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as arrears_events_12m,
                 sum(arrears_events) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as arrears_events_24m,
                 sum(doc_requests) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as doc_requests_3m,
                 sum(doc_requests) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as doc_requests_12m,
                 sum(doc_requests) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as doc_requests_24m,
                 sum(sales_calls) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as sales_calls_3m,
                 sum(sales_calls) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as sales_calls_12m,
                 sum(sales_calls) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as sales_calls_24m,
                 sum(sales_calls_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as sales_calls_il_3m,
                 sum(sales_calls_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as sales_calls_il_12m,
                 sum(sales_calls_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as sales_calls_il_24m,
                 sum(events_total) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as events_total_3m,
                 sum(events_total) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as events_total_12m,
                 sum(events_total) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as events_total_24m,
                 sum(online_quotes_motor) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as online_quotes_motor_3m,
                 sum(online_quotes_motor) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as online_quotes_motor_12m,
                 sum(online_quotes_motor) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as online_quotes_motor_24m,
                 sum(online_quotes_motor_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as online_quotes_motor_il_3m,
                 sum(online_quotes_motor_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as online_quotes_motor_il_12m,
                 sum(online_quotes_motor_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as online_quotes_motor_il_24m,
                 sum(online_quotes_home) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as online_quotes_home_3m,
                 sum(online_quotes_home) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as online_quotes_home_12m,
                 sum(online_quotes_home) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as online_quotes_home_24m,
                 sum(online_quotes_home_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as online_quotes_home_il_3m,
                 sum(online_quotes_home_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as online_quotes_home_il_12m,
                 sum(online_quotes_home_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as online_quotes_home_il_24m,
                 sum(online_quotes_van) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as online_quotes_van_3m,
                 sum(online_quotes_van) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as online_quotes_van_12m,
                 sum(online_quotes_van) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as online_quotes_van_24m,
                 sum(online_quotes_van_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as online_quotes_van_il_3m,
                 sum(online_quotes_van_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as online_quotes_van_il_12m,
                 sum(online_quotes_van_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as online_quotes_van_il_24m,
                 sum(renewal_requotes) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as renewal_requotes_3m,
                 sum(renewal_requotes) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as renewal_requotes_12m,
                 sum(renewal_requotes) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as renewal_requotes_24m,
                 sum(episodes) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as episodes_3m,
                 sum(episodes) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as episodes_12m,
                 sum(episodes) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as episodes_24m,
                 sum(episodes_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as episodes_il_3m,
                 sum(episodes_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as episodes_il_12m,
                 sum(episodes_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as episodes_il_24m,
                 sum(episodes_friction) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as episodes_friction_3m,
                 sum(episodes_friction) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as episodes_friction_12m,
                 sum(episodes_friction) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as episodes_friction_24m,
                 sum(episodes_friction_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as episodes_friction_il_3m,
                 sum(episodes_friction_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as episodes_friction_il_12m,
                 sum(episodes_friction_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as episodes_friction_il_24m,
                 sum(episodes_completed) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as episodes_completed_3m,
                 sum(episodes_completed) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as episodes_completed_12m,
                 sum(episodes_completed) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as episodes_completed_24m,
                 sum(calls_in) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as calls_in_3m,
                 sum(calls_in) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as calls_in_12m,
                 sum(calls_in) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as calls_in_24m,
                 sum(calls_in_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as calls_in_il_3m,
                 sum(calls_in_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as calls_in_il_12m,
                 sum(calls_in_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as calls_in_il_24m,
                 sum(calls_in_answered) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as calls_in_answered_3m,
                 sum(calls_in_answered) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as calls_in_answered_12m,
                 sum(calls_in_answered) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as calls_in_answered_24m,
                 sum(calls_in_abandoned) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as calls_in_abandoned_3m,
                 sum(calls_in_abandoned) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as calls_in_abandoned_12m,
                 sum(calls_in_abandoned) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as calls_in_abandoned_24m,
                 sum(calls_in_long_wait) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as calls_in_long_wait_3m,
                 sum(calls_in_long_wait) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as calls_in_long_wait_12m,
                 sum(calls_in_long_wait) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as calls_in_long_wait_24m,
                 sum(calls_in_wait_s) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as calls_in_wait_s_3m,
                 sum(calls_in_wait_s) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as calls_in_wait_s_12m,
                 sum(calls_in_wait_s) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as calls_in_wait_s_24m,
                 sum(calls_in_talk_s) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as calls_in_talk_s_3m,
                 sum(calls_in_talk_s) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as calls_in_talk_s_12m,
                 sum(calls_in_talk_s) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as calls_in_talk_s_24m,
                 sum(calls_out) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as calls_out_3m,
                 sum(calls_out) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as calls_out_12m,
                 sum(calls_out) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as calls_out_24m,
                 sum(calls_out_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as calls_out_il_3m,
                 sum(calls_out_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as calls_out_il_12m,
                 sum(calls_out_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as calls_out_il_24m,
                 sum(calls_out_talk_s) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as calls_out_talk_s_3m,
                 sum(calls_out_talk_s) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as calls_out_talk_s_12m,
                 sum(calls_out_talk_s) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as calls_out_talk_s_24m,
                 sum(mtas) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mtas_3m,
                 sum(mtas) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mtas_12m,
                 sum(mtas) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mtas_24m,
                 sum(mtas_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mtas_il_3m,
                 sum(mtas_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mtas_il_12m,
                 sum(mtas_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mtas_il_24m,
                 sum(mta_records) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mta_records_3m,
                 sum(mta_records) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mta_records_12m,
                 sum(mta_records) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mta_records_24m,
                 sum(mtas_forced) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mtas_forced_3m,
                 sum(mtas_forced) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mtas_forced_12m,
                 sum(mtas_forced) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mtas_forced_24m,
                 sum(mtas_elective) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mtas_elective_3m,
                 sum(mtas_elective) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mtas_elective_12m,
                 sum(mtas_elective) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mtas_elective_24m,
                 sum(mta_cat_cover) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mta_cat_cover_3m,
                 sum(mta_cat_cover) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mta_cat_cover_12m,
                 sum(mta_cat_cover) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mta_cat_cover_24m,
                 sum(mta_cat_drivers) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mta_cat_drivers_3m,
                 sum(mta_cat_drivers) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mta_cat_drivers_12m,
                 sum(mta_cat_drivers) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mta_cat_drivers_24m,
                 sum(mta_cat_vehicle) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mta_cat_vehicle_3m,
                 sum(mta_cat_vehicle) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mta_cat_vehicle_12m,
                 sum(mta_cat_vehicle) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mta_cat_vehicle_24m,
                 sum(mta_cat_address) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mta_cat_address_3m,
                 sum(mta_cat_address) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mta_cat_address_12m,
                 sum(mta_cat_address) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mta_cat_address_24m,
                 sum(mta_cat_licence) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mta_cat_licence_3m,
                 sum(mta_cat_licence) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mta_cat_licence_12m,
                 sum(mta_cat_licence) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mta_cat_licence_24m,
                 sum(mta_cat_use) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mta_cat_use_3m,
                 sum(mta_cat_use) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mta_cat_use_12m,
                 sum(mta_cat_use) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mta_cat_use_24m,
                 sum(mta_cat_personal) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mta_cat_personal_3m,
                 sum(mta_cat_personal) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mta_cat_personal_12m,
                 sum(mta_cat_personal) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mta_cat_personal_24m,
                 sum(loan_status_changes) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as loan_status_changes_3m,
                 sum(loan_status_changes) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as loan_status_changes_12m,
                 sum(loan_status_changes) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as loan_status_changes_24m,
                 sum(msgs_sent) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as msgs_sent_3m,
                 sum(msgs_sent) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as msgs_sent_12m,
                 sum(msgs_sent) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as msgs_sent_24m,
                 sum(msgs_sent_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as msgs_sent_il_3m,
                 sum(msgs_sent_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as msgs_sent_il_12m,
                 sum(msgs_sent_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as msgs_sent_il_24m,
                 sum(msgs_opened) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as msgs_opened_3m,
                 sum(msgs_opened) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as msgs_opened_12m,
                 sum(msgs_opened) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as msgs_opened_24m,
                 sum(msgs_opened_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as msgs_opened_il_3m,
                 sum(msgs_opened_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as msgs_opened_il_12m,
                 sum(msgs_opened_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as msgs_opened_il_24m,
                 sum(msgs_clicked) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as msgs_clicked_3m,
                 sum(msgs_clicked) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as msgs_clicked_12m,
                 sum(msgs_clicked) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as msgs_clicked_24m,
                 sum(msgs_clicked_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as msgs_clicked_il_3m,
                 sum(msgs_clicked_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as msgs_clicked_il_12m,
                 sum(msgs_clicked_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as msgs_clicked_il_24m,
                 sum(msgs_bounced) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as msgs_bounced_3m,
                 sum(msgs_bounced) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as msgs_bounced_12m,
                 sum(msgs_bounced) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as msgs_bounced_24m,
                 sum(episodes_assisted) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as episodes_assisted_3m,
                 sum(episodes_assisted) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as episodes_assisted_12m,
                 sum(episodes_assisted) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as episodes_assisted_24m,
                 sum(episodes_channel_tagged) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as episodes_channel_tagged_3m,
                 sum(episodes_channel_tagged) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as episodes_channel_tagged_12m,
                 sum(episodes_channel_tagged) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as episodes_channel_tagged_24m,
                 sum(channel_switches) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as channel_switches_3m,
                 sum(channel_switches) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as channel_switches_12m,
                 sum(channel_switches) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as channel_switches_24m,
                 sum(episodes_repeat_contact) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as episodes_repeat_contact_3m,
                 sum(episodes_repeat_contact) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as episodes_repeat_contact_12m,
                 sum(episodes_repeat_contact) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as episodes_repeat_contact_24m,
                 sum(nb_sales) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as nb_sales_3m,
                 sum(nb_sales) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as nb_sales_12m,
                 sum(nb_sales) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as nb_sales_24m,
                 sum(nb_sales_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as nb_sales_il_3m,
                 sum(nb_sales_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as nb_sales_il_12m,
                 sum(nb_sales_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as nb_sales_il_24m,
                 sum(nb_sales_web_assist) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as nb_sales_web_assist_3m,
                 sum(nb_sales_web_assist) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as nb_sales_web_assist_12m,
                 sum(nb_sales_web_assist) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as nb_sales_web_assist_24m,
                 sum(nb_sales_web_assist_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as nb_sales_web_assist_il_3m,
                 sum(nb_sales_web_assist_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as nb_sales_web_assist_il_12m,
                 sum(nb_sales_web_assist_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as nb_sales_web_assist_il_24m,
                 sum(doc_decisions) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as doc_decisions_3m,
                 sum(doc_decisions) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as doc_decisions_12m,
                 sum(doc_decisions) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as doc_decisions_24m,
                 sum(doc_rejections) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as doc_rejections_3m,
                 sum(doc_rejections) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as doc_rejections_12m,
                 sum(doc_rejections) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as doc_rejections_24m,
                 sum(doc_attempts) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as doc_attempts_3m,
                 sum(doc_attempts) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as doc_attempts_12m,
                 sum(doc_attempts) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as doc_attempts_24m,
                 sum(doc_nonreceipt) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as doc_nonreceipt_3m,
                 sum(doc_nonreceipt) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as doc_nonreceipt_12m,
                 sum(doc_nonreceipt) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as doc_nonreceipt_24m,
                 sum(doc_nonreceipt_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as doc_nonreceipt_il_3m,
                 sum(doc_nonreceipt_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as doc_nonreceipt_il_12m,
                 sum(doc_nonreceipt_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as doc_nonreceipt_il_24m,
                 sum(doc_nonreceipt_premium) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as doc_nonreceipt_premium_3m,
                 sum(doc_nonreceipt_premium) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as doc_nonreceipt_premium_12m,
                 sum(doc_nonreceipt_premium) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as doc_nonreceipt_premium_24m,
                 sum(consent_actions) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as consent_actions_3m,
                 sum(consent_actions) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as consent_actions_12m,
                 sum(consent_actions) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as consent_actions_24m,
                 sum(consent_actions_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as consent_actions_il_3m,
                 sum(consent_actions_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as consent_actions_il_12m,
                 sum(consent_actions_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as consent_actions_il_24m,
                 sum(consent_withdrawals) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as consent_withdrawals_3m,
                 sum(consent_withdrawals) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as consent_withdrawals_12m,
                 sum(consent_withdrawals) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as consent_withdrawals_24m,
                 sum(consent_withdrawals_il) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as consent_withdrawals_il_3m,
                 sum(consent_withdrawals_il) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as consent_withdrawals_il_12m,
                 sum(consent_withdrawals_il) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as consent_withdrawals_il_24m,
                 sum(mfq_quotes) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mfq_quotes_3m,
                 sum(mfq_quotes) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mfq_quotes_12m,
                 sum(mfq_quotes) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mfq_quotes_24m,
                 sum(mfq_retrievals) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mfq_retrievals_3m,
                 sum(mfq_retrievals) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mfq_retrievals_12m,
                 sum(mfq_retrievals) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mfq_retrievals_24m,
                 sum(mfq_insurers) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mfq_insurers_3m,
                 sum(mfq_insurers) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mfq_insurers_12m,
                 sum(mfq_insurers) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mfq_insurers_24m,
                 sum(mfq_returned) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mfq_returned_3m,
                 sum(mfq_returned) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mfq_returned_12m,
                 sum(mfq_returned) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mfq_returned_24m,
                 sum(mfq_cover_compared) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mfq_cover_compared_3m,
                 sum(mfq_cover_compared) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mfq_cover_compared_12m,
                 sum(mfq_cover_compared) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mfq_cover_compared_24m,
                 sum(mfq_cover_down) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mfq_cover_down_3m,
                 sum(mfq_cover_down) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mfq_cover_down_12m,
                 sum(mfq_cover_down) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mfq_cover_down_24m,
                 sum(mfq_cover_up) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mfq_cover_up_3m,
                 sum(mfq_cover_up) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mfq_cover_up_12m,
                 sum(mfq_cover_up) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mfq_cover_up_24m,
                 sum(mfq_device_tagged) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mfq_device_tagged_3m,
                 sum(mfq_device_tagged) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mfq_device_tagged_12m,
                 sum(mfq_device_tagged) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mfq_device_tagged_24m,
                 sum(mfq_device_switch) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as mfq_device_switch_3m,
                 sum(mfq_device_switch) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as mfq_device_switch_12m,
                 sum(mfq_device_switch) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as mfq_device_switch_24m,
                 sum(web_sessions) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as web_sessions_3m,
                 sum(web_sessions) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as web_sessions_12m,
                 sum(web_sessions) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as web_sessions_24m,
                 sum(web_sessions_nonquote) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as web_sessions_nonquote_3m,
                 sum(web_sessions_nonquote) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as web_sessions_nonquote_12m,
                 sum(web_sessions_nonquote) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as web_sessions_nonquote_24m,
                 sum(web_page_views) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as web_page_views_3m,
                 sum(web_page_views) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as web_page_views_12m,
                 sum(web_page_views) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as web_page_views_24m,
                 sum(web_content_views) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as web_content_views_3m,
                 sum(web_content_views) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as web_content_views_12m,
                 sum(web_content_views) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as web_content_views_24m,
                 sum(web_other_product_sessions) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as web_other_product_sessions_3m,
                 sum(web_other_product_sessions) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as web_other_product_sessions_12m,
                 sum(web_other_product_sessions) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as web_other_product_sessions_24m,
                 sum(calls_other_queues) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as calls_other_queues_3m,
                 sum(calls_other_queues) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as calls_other_queues_12m,
                 sum(calls_other_queues) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as calls_other_queues_24m,
                 sum(friction_all) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_all_3m,
                 sum(friction_all) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_all_12m,
                 sum(friction_all) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_all_24m,
                 sum(friction_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_resolved_3m,
                 sum(friction_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_resolved_12m,
                 sum(friction_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_resolved_24m,
                 sum(friction_payment) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_payment_3m,
                 sum(friction_payment) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_payment_12m,
                 sum(friction_payment) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_payment_24m,
                 sum(friction_payment_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_payment_resolved_3m,
                 sum(friction_payment_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_payment_resolved_12m,
                 sum(friction_payment_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_payment_resolved_24m,
                 sum(friction_login) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_login_3m,
                 sum(friction_login) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_login_12m,
                 sum(friction_login) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_login_24m,
                 sum(friction_login_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_login_resolved_3m,
                 sum(friction_login_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_login_resolved_12m,
                 sum(friction_login_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_login_resolved_24m,
                 sum(friction_contact) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_contact_3m,
                 sum(friction_contact) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_contact_12m,
                 sum(friction_contact) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_contact_24m,
                 sum(friction_contact_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_contact_resolved_3m,
                 sum(friction_contact_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_contact_resolved_12m,
                 sum(friction_contact_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_contact_resolved_24m,
                 sum(friction_documents) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_documents_3m,
                 sum(friction_documents) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_documents_12m,
                 sum(friction_documents) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_documents_24m,
                 sum(friction_documents_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_documents_resolved_3m,
                 sum(friction_documents_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_documents_resolved_12m,
                 sum(friction_documents_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_documents_resolved_24m,
                 sum(friction_quote) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_quote_3m,
                 sum(friction_quote) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_quote_12m,
                 sum(friction_quote) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_quote_24m,
                 sum(friction_quote_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_quote_resolved_3m,
                 sum(friction_quote_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_quote_resolved_12m,
                 sum(friction_quote_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_quote_resolved_24m,
                 sum(friction_process) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_process_3m,
                 sum(friction_process) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_process_12m,
                 sum(friction_process) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_process_24m,
                 sum(friction_process_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_process_resolved_3m,
                 sum(friction_process_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_process_resolved_12m,
                 sum(friction_process_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_process_resolved_24m,
                 sum(friction_other) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_other_3m,
                 sum(friction_other) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_other_12m,
                 sum(friction_other) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_other_24m,
                 sum(friction_other_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_other_resolved_3m,
                 sum(friction_other_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_other_resolved_12m,
                 sum(friction_other_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_other_resolved_24m,
                 sum(offers_sent) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as offers_sent_3m,
                 sum(offers_sent) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as offers_sent_12m,
                 sum(offers_sent) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as offers_sent_24m,
                 sum(offers_clicked) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as offers_clicked_3m,
                 sum(offers_clicked) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as offers_clicked_12m,
                 sum(offers_clicked) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as offers_clicked_24m,
                 sum(offer_clicks_nb) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as offer_clicks_nb_3m,
                 sum(offer_clicks_nb) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as offer_clicks_nb_12m,
                 sum(offer_clicks_nb) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as offer_clicks_nb_24m,
                 sum(offer_clicks_renewal) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as offer_clicks_renewal_3m,
                 sum(offer_clicks_renewal) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as offer_clicks_renewal_12m,
                 sum(offer_clicks_renewal) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as offer_clicks_renewal_24m,
                 sum(friction_v3_eligible) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_eligible_3m,
                 sum(friction_v3_eligible) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_eligible_12m,
                 sum(friction_v3_eligible) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_eligible_24m,
                 sum(friction_v3_affected) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_affected_3m,
                 sum(friction_v3_affected) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_affected_12m,
                 sum(friction_v3_affected) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_affected_24m,
                 sum(friction_v3_res_known) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_res_known_3m,
                 sum(friction_v3_res_known) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_res_known_12m,
                 sum(friction_v3_res_known) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_res_known_24m,
                 sum(friction_v3_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_resolved_3m,
                 sum(friction_v3_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_resolved_12m,
                 sum(friction_v3_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_resolved_24m,
                 sum(friction_v3_renewal_eligible) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_renewal_eligible_3m,
                 sum(friction_v3_renewal_eligible) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_renewal_eligible_12m,
                 sum(friction_v3_renewal_eligible) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_renewal_eligible_24m,
                 sum(friction_v3_renewal_affected) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_renewal_affected_3m,
                 sum(friction_v3_renewal_affected) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_renewal_affected_12m,
                 sum(friction_v3_renewal_affected) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_renewal_affected_24m,
                 sum(friction_v3_renewal_res_known) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_renewal_res_known_3m,
                 sum(friction_v3_renewal_res_known) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_renewal_res_known_12m,
                 sum(friction_v3_renewal_res_known) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_renewal_res_known_24m,
                 sum(friction_v3_renewal_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_renewal_resolved_3m,
                 sum(friction_v3_renewal_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_renewal_resolved_12m,
                 sum(friction_v3_renewal_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_renewal_resolved_24m,
                 sum(friction_v3_firstyear_eligible) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_firstyear_eligible_3m,
                 sum(friction_v3_firstyear_eligible) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_firstyear_eligible_12m,
                 sum(friction_v3_firstyear_eligible) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_firstyear_eligible_24m,
                 sum(friction_v3_firstyear_affected) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_firstyear_affected_3m,
                 sum(friction_v3_firstyear_affected) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_firstyear_affected_12m,
                 sum(friction_v3_firstyear_affected) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_firstyear_affected_24m,
                 sum(friction_v3_firstyear_res_known) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_firstyear_res_known_3m,
                 sum(friction_v3_firstyear_res_known) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_firstyear_res_known_12m,
                 sum(friction_v3_firstyear_res_known) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_firstyear_res_known_24m,
                 sum(friction_v3_firstyear_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_firstyear_resolved_3m,
                 sum(friction_v3_firstyear_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_firstyear_resolved_12m,
                 sum(friction_v3_firstyear_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_firstyear_resolved_24m,
                 sum(friction_v3_change_eligible) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_change_eligible_3m,
                 sum(friction_v3_change_eligible) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_change_eligible_12m,
                 sum(friction_v3_change_eligible) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_change_eligible_24m,
                 sum(friction_v3_change_affected) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_change_affected_3m,
                 sum(friction_v3_change_affected) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_change_affected_12m,
                 sum(friction_v3_change_affected) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_change_affected_24m,
                 sum(friction_v3_change_res_known) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_change_res_known_3m,
                 sum(friction_v3_change_res_known) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_change_res_known_12m,
                 sum(friction_v3_change_res_known) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_change_res_known_24m,
                 sum(friction_v3_change_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_change_resolved_3m,
                 sum(friction_v3_change_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_change_resolved_12m,
                 sum(friction_v3_change_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_change_resolved_24m,
                 sum(friction_v3_docreq_eligible) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_docreq_eligible_3m,
                 sum(friction_v3_docreq_eligible) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_docreq_eligible_12m,
                 sum(friction_v3_docreq_eligible) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_docreq_eligible_24m,
                 sum(friction_v3_docreq_affected) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_docreq_affected_3m,
                 sum(friction_v3_docreq_affected) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_docreq_affected_12m,
                 sum(friction_v3_docreq_affected) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_docreq_affected_24m,
                 sum(friction_v3_docreq_res_known) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_docreq_res_known_3m,
                 sum(friction_v3_docreq_res_known) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_docreq_res_known_12m,
                 sum(friction_v3_docreq_res_known) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_docreq_res_known_24m,
                 sum(friction_v3_docreq_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as friction_v3_docreq_resolved_3m,
                 sum(friction_v3_docreq_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as friction_v3_docreq_resolved_12m,
                 sum(friction_v3_docreq_resolved) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as friction_v3_docreq_resolved_24m,
                 sum(months_held) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as months_held_3m,
                 sum(months_held) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as months_held_12m,
                 sum(months_held) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as months_held_24m,
                 sum(renewals_due) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as renewals_due_3m,
                 sum(renewals_due) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as renewals_due_12m,
                 sum(renewals_due) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as renewals_due_24m,
                 sum(motor_renewals_due) over (partition by SCV_Customer_Key order by MonthNo range between 2 preceding and current row) as motor_renewals_due_3m,
                 sum(motor_renewals_due) over (partition by SCV_Customer_Key order by MonthNo range between 11 preceding and current row) as motor_renewals_due_12m,
                 sum(motor_renewals_due) over (partition by SCV_Customer_Key order by MonthNo range between 23 preceding and current row) as motor_renewals_due_24m,
                 sum(calls_in) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 3 following) as calls_in_next3m,
                 sum(calls_in) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 12 following) as calls_in_next12m,
                 sum(calls_in_talk_s) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 3 following) as calls_in_talk_s_next3m,
                 sum(calls_in_talk_s) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 12 following) as calls_in_talk_s_next12m,
                 sum(calls_out) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 3 following) as calls_out_next3m,
                 sum(calls_out) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 12 following) as calls_out_next12m,
                 sum(calls_out_talk_s) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 3 following) as calls_out_talk_s_next3m,
                 sum(calls_out_talk_s) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 12 following) as calls_out_talk_s_next12m,
                 sum(inlife_contacts) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 3 following) as inlife_contacts_next3m,
                 sum(inlife_contacts) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 12 following) as inlife_contacts_next12m,
                 sum(mtas) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 3 following) as mtas_next3m,
                 sum(mtas) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 12 following) as mtas_next12m,
                 sum(doc_requests) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 3 following) as doc_requests_next3m,
                 sum(doc_requests) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 12 following) as doc_requests_next12m,
                 sum(chases) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 3 following) as chases_next3m,
                 sum(chases) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 12 following) as chases_next12m,
                 sum(failed_payments) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 3 following) as failed_payments_next3m,
                 sum(failed_payments) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 12 following) as failed_payments_next12m,
                 sum(friction_events) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 3 following) as friction_events_next3m,
                 sum(friction_events) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 12 following) as friction_events_next12m,
                 sum(arrears_events) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 3 following) as arrears_events_next3m,
                 sum(arrears_events) over (partition by SCV_Customer_Key order by MonthNo range between 1 following and 12 following) as arrears_events_next12m
         from    ${beh.stg}.beh_grid) w
join    ${beh.target}.CustomerMonth c
        on  c.SCV_Customer_Key = w.SCV_Customer_Key
        and c.MonthNo = w.MonthNo


## beh_behaviour_base


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_behaviour_base                                                            */
/*  Description : The windowed counts with the coverage rule applied: null where the window starts  */
/*                before the source's coverage start, else the count (0 when nothing happened).     */
/*  Reads       : stg.beh_rolling, settings cov_*                                                   */
/*  Key         : SCV_Customer_Key + MonthEndDate                                                   */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_behaviour_base as
select  r.SCV_Customer_Key,
        r.MonthEndDate,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_events}') then coalesce(r.prompts_3m, 0) end as prompts_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_events}') then coalesce(r.prompts_12m, 0) end as prompts_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_events}') then coalesce(r.prompts_24m, 0) end as prompts_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.ren_cust_actions_3m, 0) end as ren_cust_actions_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.ren_cust_actions_12m, 0) end as ren_cust_actions_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.ren_cust_actions_24m, 0) end as ren_cust_actions_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.inlife_contacts_3m, 0) end as inlife_contacts_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.inlife_contacts_12m, 0) end as inlife_contacts_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.inlife_contacts_24m, 0) end as inlife_contacts_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_events_3m, 0) end as friction_events_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_events_12m, 0) end as friction_events_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_events_24m, 0) end as friction_events_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_events_il_3m, 0) end as friction_events_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_events_il_12m, 0) end as friction_events_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_events_il_24m, 0) end as friction_events_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mychill}') then coalesce(r.doc_friction_events_3m, 0) end as doc_friction_events_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mychill}') then coalesce(r.doc_friction_events_12m, 0) end as doc_friction_events_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mychill}') then coalesce(r.doc_friction_events_24m, 0) end as doc_friction_events_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mychill}') then coalesce(r.chases_3m, 0) end as chases_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mychill}') then coalesce(r.chases_12m, 0) end as chases_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mychill}') then coalesce(r.chases_24m, 0) end as chases_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mychill}') then coalesce(r.chases_il_3m, 0) end as chases_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mychill}') then coalesce(r.chases_il_12m, 0) end as chases_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mychill}') then coalesce(r.chases_il_24m, 0) end as chases_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.hist_start}') then coalesce(r.concessions_3m, 0) end as concessions_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.hist_start}') then coalesce(r.concessions_12m, 0) end as concessions_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.hist_start}') then coalesce(r.concessions_24m, 0) end as concessions_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_events}') then coalesce(r.failed_payments_3m, 0) end as failed_payments_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_events}') then coalesce(r.failed_payments_12m, 0) end as failed_payments_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_events}') then coalesce(r.failed_payments_24m, 0) end as failed_payments_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_events}') then coalesce(r.failed_payments_il_3m, 0) end as failed_payments_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_events}') then coalesce(r.failed_payments_il_12m, 0) end as failed_payments_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_events}') then coalesce(r.failed_payments_il_24m, 0) end as failed_payments_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_events}') then coalesce(r.failed_logins_3m, 0) end as failed_logins_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_events}') then coalesce(r.failed_logins_12m, 0) end as failed_logins_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_events}') then coalesce(r.failed_logins_24m, 0) end as failed_logins_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_arrears}') then coalesce(r.arrears_events_3m, 0) end as arrears_events_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_arrears}') then coalesce(r.arrears_events_12m, 0) end as arrears_events_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_arrears}') then coalesce(r.arrears_events_24m, 0) end as arrears_events_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.doc_requests_3m, 0) end as doc_requests_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.doc_requests_12m, 0) end as doc_requests_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.doc_requests_24m, 0) end as doc_requests_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.sales_calls_3m, 0) end as sales_calls_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.sales_calls_12m, 0) end as sales_calls_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.sales_calls_24m, 0) end as sales_calls_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.sales_calls_il_3m, 0) end as sales_calls_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.sales_calls_il_12m, 0) end as sales_calls_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.sales_calls_il_24m, 0) end as sales_calls_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_events}') then coalesce(r.events_total_3m, 0) end as events_total_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_events}') then coalesce(r.events_total_12m, 0) end as events_total_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_events}') then coalesce(r.events_total_24m, 0) end as events_total_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_motor_3m, 0) end as online_quotes_motor_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_motor_12m, 0) end as online_quotes_motor_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_motor_24m, 0) end as online_quotes_motor_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_motor_il_3m, 0) end as online_quotes_motor_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_motor_il_12m, 0) end as online_quotes_motor_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_motor_il_24m, 0) end as online_quotes_motor_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_home_3m, 0) end as online_quotes_home_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_home_12m, 0) end as online_quotes_home_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_home_24m, 0) end as online_quotes_home_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_home_il_3m, 0) end as online_quotes_home_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_home_il_12m, 0) end as online_quotes_home_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_home_il_24m, 0) end as online_quotes_home_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_van_3m, 0) end as online_quotes_van_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_van_12m, 0) end as online_quotes_van_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_van_24m, 0) end as online_quotes_van_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_van_il_3m, 0) end as online_quotes_van_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_van_il_12m, 0) end as online_quotes_van_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.online_quotes_van_il_24m, 0) end as online_quotes_van_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.renewal_requotes_3m, 0) end as renewal_requotes_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.renewal_requotes_12m, 0) end as renewal_requotes_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_quotes}') then coalesce(r.renewal_requotes_24m, 0) end as renewal_requotes_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_events}') then coalesce(r.episodes_3m, 0) end as episodes_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_events}') then coalesce(r.episodes_12m, 0) end as episodes_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_events}') then coalesce(r.episodes_24m, 0) end as episodes_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_events}') then coalesce(r.episodes_il_3m, 0) end as episodes_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_events}') then coalesce(r.episodes_il_12m, 0) end as episodes_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_events}') then coalesce(r.episodes_il_24m, 0) end as episodes_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.episodes_friction_3m, 0) end as episodes_friction_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.episodes_friction_12m, 0) end as episodes_friction_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.episodes_friction_24m, 0) end as episodes_friction_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.episodes_friction_il_3m, 0) end as episodes_friction_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.episodes_friction_il_12m, 0) end as episodes_friction_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.episodes_friction_il_24m, 0) end as episodes_friction_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_events}') then coalesce(r.episodes_completed_3m, 0) end as episodes_completed_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_events}') then coalesce(r.episodes_completed_12m, 0) end as episodes_completed_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_events}') then coalesce(r.episodes_completed_24m, 0) end as episodes_completed_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_3m, 0) end as calls_in_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_12m, 0) end as calls_in_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_24m, 0) end as calls_in_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_il_3m, 0) end as calls_in_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_il_12m, 0) end as calls_in_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_il_24m, 0) end as calls_in_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_answered_3m, 0) end as calls_in_answered_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_answered_12m, 0) end as calls_in_answered_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_answered_24m, 0) end as calls_in_answered_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_abandoned_3m, 0) end as calls_in_abandoned_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_abandoned_12m, 0) end as calls_in_abandoned_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_abandoned_24m, 0) end as calls_in_abandoned_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_long_wait_3m, 0) end as calls_in_long_wait_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_long_wait_12m, 0) end as calls_in_long_wait_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_long_wait_24m, 0) end as calls_in_long_wait_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_wait_s_3m, 0) end as calls_in_wait_s_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_wait_s_12m, 0) end as calls_in_wait_s_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_wait_s_24m, 0) end as calls_in_wait_s_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_talk_s_3m, 0) end as calls_in_talk_s_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_talk_s_12m, 0) end as calls_in_talk_s_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_talk_s_24m, 0) end as calls_in_talk_s_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_out_3m, 0) end as calls_out_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_out_12m, 0) end as calls_out_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_out_24m, 0) end as calls_out_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_out_il_3m, 0) end as calls_out_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_out_il_12m, 0) end as calls_out_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_out_il_24m, 0) end as calls_out_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_out_talk_s_3m, 0) end as calls_out_talk_s_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_out_talk_s_12m, 0) end as calls_out_talk_s_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_out_talk_s_24m, 0) end as calls_out_talk_s_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mta_count}') then coalesce(r.mtas_3m, 0) end as mtas_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mta_count}') then coalesce(r.mtas_12m, 0) end as mtas_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mta_count}') then coalesce(r.mtas_24m, 0) end as mtas_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mta_count}') then coalesce(r.mtas_il_3m, 0) end as mtas_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mta_count}') then coalesce(r.mtas_il_12m, 0) end as mtas_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mta_count}') then coalesce(r.mtas_il_24m, 0) end as mtas_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_records_3m, 0) end as mta_records_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_records_12m, 0) end as mta_records_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_records_24m, 0) end as mta_records_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mtas_forced_3m, 0) end as mtas_forced_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mtas_forced_12m, 0) end as mtas_forced_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mtas_forced_24m, 0) end as mtas_forced_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mtas_elective_3m, 0) end as mtas_elective_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mtas_elective_12m, 0) end as mtas_elective_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mtas_elective_24m, 0) end as mtas_elective_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_cover_3m, 0) end as mta_cat_cover_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_cover_12m, 0) end as mta_cat_cover_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_cover_24m, 0) end as mta_cat_cover_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_drivers_3m, 0) end as mta_cat_drivers_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_drivers_12m, 0) end as mta_cat_drivers_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_drivers_24m, 0) end as mta_cat_drivers_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_vehicle_3m, 0) end as mta_cat_vehicle_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_vehicle_12m, 0) end as mta_cat_vehicle_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_vehicle_24m, 0) end as mta_cat_vehicle_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_address_3m, 0) end as mta_cat_address_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_address_12m, 0) end as mta_cat_address_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_address_24m, 0) end as mta_cat_address_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_licence_3m, 0) end as mta_cat_licence_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_licence_12m, 0) end as mta_cat_licence_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_licence_24m, 0) end as mta_cat_licence_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_use_3m, 0) end as mta_cat_use_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_use_12m, 0) end as mta_cat_use_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_use_24m, 0) end as mta_cat_use_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_personal_3m, 0) end as mta_cat_personal_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_personal_12m, 0) end as mta_cat_personal_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mta}') then coalesce(r.mta_cat_personal_24m, 0) end as mta_cat_personal_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_loan}') then coalesce(r.loan_status_changes_3m, 0) end as loan_status_changes_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_loan}') then coalesce(r.loan_status_changes_12m, 0) end as loan_status_changes_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_loan}') then coalesce(r.loan_status_changes_24m, 0) end as loan_status_changes_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_sent_3m, 0) end as msgs_sent_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_sent_12m, 0) end as msgs_sent_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_sent_24m, 0) end as msgs_sent_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_sent_il_3m, 0) end as msgs_sent_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_sent_il_12m, 0) end as msgs_sent_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_sent_il_24m, 0) end as msgs_sent_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_opened_3m, 0) end as msgs_opened_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_opened_12m, 0) end as msgs_opened_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_opened_24m, 0) end as msgs_opened_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_opened_il_3m, 0) end as msgs_opened_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_opened_il_12m, 0) end as msgs_opened_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_opened_il_24m, 0) end as msgs_opened_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_clicked_3m, 0) end as msgs_clicked_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_clicked_12m, 0) end as msgs_clicked_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_clicked_24m, 0) end as msgs_clicked_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_clicked_il_3m, 0) end as msgs_clicked_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_clicked_il_12m, 0) end as msgs_clicked_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_clicked_il_24m, 0) end as msgs_clicked_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_bounced_3m, 0) end as msgs_bounced_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_bounced_12m, 0) end as msgs_bounced_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_comms}') then coalesce(r.msgs_bounced_24m, 0) end as msgs_bounced_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.episodes_assisted_3m, 0) end as episodes_assisted_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.episodes_assisted_12m, 0) end as episodes_assisted_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.episodes_assisted_24m, 0) end as episodes_assisted_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.episodes_channel_tagged_3m, 0) end as episodes_channel_tagged_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.episodes_channel_tagged_12m, 0) end as episodes_channel_tagged_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.episodes_channel_tagged_24m, 0) end as episodes_channel_tagged_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.channel_switches_3m, 0) end as channel_switches_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.channel_switches_12m, 0) end as channel_switches_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.channel_switches_24m, 0) end as channel_switches_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.episodes_repeat_contact_3m, 0) end as episodes_repeat_contact_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.episodes_repeat_contact_12m, 0) end as episodes_repeat_contact_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.episodes_repeat_contact_24m, 0) end as episodes_repeat_contact_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.hist_start}') then coalesce(r.nb_sales_3m, 0) end as nb_sales_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.hist_start}') then coalesce(r.nb_sales_12m, 0) end as nb_sales_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.hist_start}') then coalesce(r.nb_sales_24m, 0) end as nb_sales_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.hist_start}') then coalesce(r.nb_sales_il_3m, 0) end as nb_sales_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.hist_start}') then coalesce(r.nb_sales_il_12m, 0) end as nb_sales_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.hist_start}') then coalesce(r.nb_sales_il_24m, 0) end as nb_sales_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.hist_start}') then coalesce(r.nb_sales_web_assist_3m, 0) end as nb_sales_web_assist_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.hist_start}') then coalesce(r.nb_sales_web_assist_12m, 0) end as nb_sales_web_assist_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.hist_start}') then coalesce(r.nb_sales_web_assist_24m, 0) end as nb_sales_web_assist_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.hist_start}') then coalesce(r.nb_sales_web_assist_il_3m, 0) end as nb_sales_web_assist_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.hist_start}') then coalesce(r.nb_sales_web_assist_il_12m, 0) end as nb_sales_web_assist_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.hist_start}') then coalesce(r.nb_sales_web_assist_il_24m, 0) end as nb_sales_web_assist_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_docwf}') then coalesce(r.doc_decisions_3m, 0) end as doc_decisions_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_docwf}') then coalesce(r.doc_decisions_12m, 0) end as doc_decisions_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_docwf}') then coalesce(r.doc_decisions_24m, 0) end as doc_decisions_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_docwf}') then coalesce(r.doc_rejections_3m, 0) end as doc_rejections_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_docwf}') then coalesce(r.doc_rejections_12m, 0) end as doc_rejections_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_docwf}') then coalesce(r.doc_rejections_24m, 0) end as doc_rejections_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_docwf}') then coalesce(r.doc_attempts_3m, 0) end as doc_attempts_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_docwf}') then coalesce(r.doc_attempts_12m, 0) end as doc_attempts_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_docwf}') then coalesce(r.doc_attempts_24m, 0) end as doc_attempts_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.doc_nonreceipt_3m, 0) end as doc_nonreceipt_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.doc_nonreceipt_12m, 0) end as doc_nonreceipt_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.doc_nonreceipt_24m, 0) end as doc_nonreceipt_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.doc_nonreceipt_il_3m, 0) end as doc_nonreceipt_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.doc_nonreceipt_il_12m, 0) end as doc_nonreceipt_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.doc_nonreceipt_il_24m, 0) end as doc_nonreceipt_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.doc_nonreceipt_premium_3m, 0) end as doc_nonreceipt_premium_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.doc_nonreceipt_premium_12m, 0) end as doc_nonreceipt_premium_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.doc_nonreceipt_premium_24m, 0) end as doc_nonreceipt_premium_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_consent}') then coalesce(r.consent_actions_3m, 0) end as consent_actions_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_consent}') then coalesce(r.consent_actions_12m, 0) end as consent_actions_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_consent}') then coalesce(r.consent_actions_24m, 0) end as consent_actions_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_consent}') then coalesce(r.consent_actions_il_3m, 0) end as consent_actions_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_consent}') then coalesce(r.consent_actions_il_12m, 0) end as consent_actions_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_consent}') then coalesce(r.consent_actions_il_24m, 0) end as consent_actions_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_consent}') then coalesce(r.consent_withdrawals_3m, 0) end as consent_withdrawals_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_consent}') then coalesce(r.consent_withdrawals_12m, 0) end as consent_withdrawals_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_consent}') then coalesce(r.consent_withdrawals_24m, 0) end as consent_withdrawals_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_consent}') then coalesce(r.consent_withdrawals_il_3m, 0) end as consent_withdrawals_il_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_consent}') then coalesce(r.consent_withdrawals_il_12m, 0) end as consent_withdrawals_il_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_consent}') then coalesce(r.consent_withdrawals_il_24m, 0) end as consent_withdrawals_il_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_quotes_3m, 0) end as mfq_quotes_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_quotes_12m, 0) end as mfq_quotes_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_quotes_24m, 0) end as mfq_quotes_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_retrievals_3m, 0) end as mfq_retrievals_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_retrievals_12m, 0) end as mfq_retrievals_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_retrievals_24m, 0) end as mfq_retrievals_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_insurers_3m, 0) end as mfq_insurers_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_insurers_12m, 0) end as mfq_insurers_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_insurers_24m, 0) end as mfq_insurers_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_returned_3m, 0) end as mfq_returned_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_returned_12m, 0) end as mfq_returned_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_returned_24m, 0) end as mfq_returned_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_cover_compared_3m, 0) end as mfq_cover_compared_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_cover_compared_12m, 0) end as mfq_cover_compared_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_cover_compared_24m, 0) end as mfq_cover_compared_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_cover_down_3m, 0) end as mfq_cover_down_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_cover_down_12m, 0) end as mfq_cover_down_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_cover_down_24m, 0) end as mfq_cover_down_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_cover_up_3m, 0) end as mfq_cover_up_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_cover_up_12m, 0) end as mfq_cover_up_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mfq}') then coalesce(r.mfq_cover_up_24m, 0) end as mfq_cover_up_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mfq_device}') then coalesce(r.mfq_device_tagged_3m, 0) end as mfq_device_tagged_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mfq_device}') then coalesce(r.mfq_device_tagged_12m, 0) end as mfq_device_tagged_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mfq_device}') then coalesce(r.mfq_device_tagged_24m, 0) end as mfq_device_tagged_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_mfq_device}') then coalesce(r.mfq_device_switch_3m, 0) end as mfq_device_switch_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_mfq_device}') then coalesce(r.mfq_device_switch_12m, 0) end as mfq_device_switch_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_mfq_device}') then coalesce(r.mfq_device_switch_24m, 0) end as mfq_device_switch_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_web}') then coalesce(r.web_sessions_3m, 0) end as web_sessions_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_web}') then coalesce(r.web_sessions_12m, 0) end as web_sessions_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_web}') then coalesce(r.web_sessions_24m, 0) end as web_sessions_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_web}') then coalesce(r.web_sessions_nonquote_3m, 0) end as web_sessions_nonquote_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_web}') then coalesce(r.web_sessions_nonquote_12m, 0) end as web_sessions_nonquote_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_web}') then coalesce(r.web_sessions_nonquote_24m, 0) end as web_sessions_nonquote_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_web}') then coalesce(r.web_page_views_3m, 0) end as web_page_views_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_web}') then coalesce(r.web_page_views_12m, 0) end as web_page_views_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_web}') then coalesce(r.web_page_views_24m, 0) end as web_page_views_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_web}') then coalesce(r.web_content_views_3m, 0) end as web_content_views_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_web}') then coalesce(r.web_content_views_12m, 0) end as web_content_views_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_web}') then coalesce(r.web_content_views_24m, 0) end as web_content_views_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_web}') then coalesce(r.web_other_product_sessions_3m, 0) end as web_other_product_sessions_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_web}') then coalesce(r.web_other_product_sessions_12m, 0) end as web_other_product_sessions_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_web}') then coalesce(r.web_other_product_sessions_24m, 0) end as web_other_product_sessions_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_tcall}') then coalesce(r.calls_other_queues_3m, 0) end as calls_other_queues_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_tcall}') then coalesce(r.calls_other_queues_12m, 0) end as calls_other_queues_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_tcall}') then coalesce(r.calls_other_queues_24m, 0) end as calls_other_queues_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_all_3m, 0) end as friction_all_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_all_12m, 0) end as friction_all_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_all_24m, 0) end as friction_all_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_resolved_3m, 0) end as friction_resolved_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_resolved_12m, 0) end as friction_resolved_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_resolved_24m, 0) end as friction_resolved_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_payment_3m, 0) end as friction_payment_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_payment_12m, 0) end as friction_payment_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_payment_24m, 0) end as friction_payment_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_payment_resolved_3m, 0) end as friction_payment_resolved_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_payment_resolved_12m, 0) end as friction_payment_resolved_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_payment_resolved_24m, 0) end as friction_payment_resolved_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_login_3m, 0) end as friction_login_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_login_12m, 0) end as friction_login_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_login_24m, 0) end as friction_login_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_login_resolved_3m, 0) end as friction_login_resolved_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_login_resolved_12m, 0) end as friction_login_resolved_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_login_resolved_24m, 0) end as friction_login_resolved_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_contact_3m, 0) end as friction_contact_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_contact_12m, 0) end as friction_contact_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_contact_24m, 0) end as friction_contact_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_contact_resolved_3m, 0) end as friction_contact_resolved_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_contact_resolved_12m, 0) end as friction_contact_resolved_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_contact_resolved_24m, 0) end as friction_contact_resolved_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_documents_3m, 0) end as friction_documents_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_documents_12m, 0) end as friction_documents_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_documents_24m, 0) end as friction_documents_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_documents_resolved_3m, 0) end as friction_documents_resolved_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_documents_resolved_12m, 0) end as friction_documents_resolved_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_documents_resolved_24m, 0) end as friction_documents_resolved_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_quote_3m, 0) end as friction_quote_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_quote_12m, 0) end as friction_quote_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_quote_24m, 0) end as friction_quote_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_quote_resolved_3m, 0) end as friction_quote_resolved_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_quote_resolved_12m, 0) end as friction_quote_resolved_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_quote_resolved_24m, 0) end as friction_quote_resolved_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_process_3m, 0) end as friction_process_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_process_12m, 0) end as friction_process_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_process_24m, 0) end as friction_process_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_process_resolved_3m, 0) end as friction_process_resolved_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_process_resolved_12m, 0) end as friction_process_resolved_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_process_resolved_24m, 0) end as friction_process_resolved_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_other_3m, 0) end as friction_other_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_other_12m, 0) end as friction_other_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_other_24m, 0) end as friction_other_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_other_resolved_3m, 0) end as friction_other_resolved_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_other_resolved_12m, 0) end as friction_other_resolved_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_other_resolved_24m, 0) end as friction_other_resolved_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_comms}') then coalesce(r.offers_sent_3m, 0) end as offers_sent_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_comms}') then coalesce(r.offers_sent_12m, 0) end as offers_sent_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_comms}') then coalesce(r.offers_sent_24m, 0) end as offers_sent_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_comms}') then coalesce(r.offers_clicked_3m, 0) end as offers_clicked_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_comms}') then coalesce(r.offers_clicked_12m, 0) end as offers_clicked_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_comms}') then coalesce(r.offers_clicked_24m, 0) end as offers_clicked_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_comms}') then coalesce(r.offer_clicks_nb_3m, 0) end as offer_clicks_nb_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_comms}') then coalesce(r.offer_clicks_nb_12m, 0) end as offer_clicks_nb_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_comms}') then coalesce(r.offer_clicks_nb_24m, 0) end as offer_clicks_nb_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_comms}') then coalesce(r.offer_clicks_renewal_3m, 0) end as offer_clicks_renewal_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_comms}') then coalesce(r.offer_clicks_renewal_12m, 0) end as offer_clicks_renewal_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_comms}') then coalesce(r.offer_clicks_renewal_24m, 0) end as offer_clicks_renewal_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_eligible_3m, 0) end as friction_v3_eligible_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_eligible_12m, 0) end as friction_v3_eligible_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_eligible_24m, 0) end as friction_v3_eligible_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_affected_3m, 0) end as friction_v3_affected_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_affected_12m, 0) end as friction_v3_affected_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_affected_24m, 0) end as friction_v3_affected_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_res_known_3m, 0) end as friction_v3_res_known_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_res_known_12m, 0) end as friction_v3_res_known_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_res_known_24m, 0) end as friction_v3_res_known_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_resolved_3m, 0) end as friction_v3_resolved_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_resolved_12m, 0) end as friction_v3_resolved_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_resolved_24m, 0) end as friction_v3_resolved_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_renewal_eligible_3m, 0) end as friction_v3_renewal_eligible_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_renewal_eligible_12m, 0) end as friction_v3_renewal_eligible_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_renewal_eligible_24m, 0) end as friction_v3_renewal_eligible_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_renewal_affected_3m, 0) end as friction_v3_renewal_affected_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_renewal_affected_12m, 0) end as friction_v3_renewal_affected_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_renewal_affected_24m, 0) end as friction_v3_renewal_affected_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_renewal_res_known_3m, 0) end as friction_v3_renewal_res_known_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_renewal_res_known_12m, 0) end as friction_v3_renewal_res_known_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_renewal_res_known_24m, 0) end as friction_v3_renewal_res_known_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_renewal_resolved_3m, 0) end as friction_v3_renewal_resolved_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_renewal_resolved_12m, 0) end as friction_v3_renewal_resolved_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_renewal_resolved_24m, 0) end as friction_v3_renewal_resolved_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_firstyear_eligible_3m, 0) end as friction_v3_firstyear_eligible_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_firstyear_eligible_12m, 0) end as friction_v3_firstyear_eligible_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_firstyear_eligible_24m, 0) end as friction_v3_firstyear_eligible_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_firstyear_affected_3m, 0) end as friction_v3_firstyear_affected_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_firstyear_affected_12m, 0) end as friction_v3_firstyear_affected_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_firstyear_affected_24m, 0) end as friction_v3_firstyear_affected_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_firstyear_res_known_3m, 0) end as friction_v3_firstyear_res_known_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_firstyear_res_known_12m, 0) end as friction_v3_firstyear_res_known_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_firstyear_res_known_24m, 0) end as friction_v3_firstyear_res_known_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_firstyear_resolved_3m, 0) end as friction_v3_firstyear_resolved_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_firstyear_resolved_12m, 0) end as friction_v3_firstyear_resolved_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_firstyear_resolved_24m, 0) end as friction_v3_firstyear_resolved_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_change_eligible_3m, 0) end as friction_v3_change_eligible_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_change_eligible_12m, 0) end as friction_v3_change_eligible_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_change_eligible_24m, 0) end as friction_v3_change_eligible_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_change_affected_3m, 0) end as friction_v3_change_affected_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_change_affected_12m, 0) end as friction_v3_change_affected_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_change_affected_24m, 0) end as friction_v3_change_affected_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_change_res_known_3m, 0) end as friction_v3_change_res_known_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_change_res_known_12m, 0) end as friction_v3_change_res_known_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_change_res_known_24m, 0) end as friction_v3_change_res_known_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_change_resolved_3m, 0) end as friction_v3_change_resolved_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_change_resolved_12m, 0) end as friction_v3_change_resolved_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_change_resolved_24m, 0) end as friction_v3_change_resolved_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_docreq_eligible_3m, 0) end as friction_v3_docreq_eligible_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_docreq_eligible_12m, 0) end as friction_v3_docreq_eligible_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_docreq_eligible_24m, 0) end as friction_v3_docreq_eligible_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_docreq_affected_3m, 0) end as friction_v3_docreq_affected_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_docreq_affected_12m, 0) end as friction_v3_docreq_affected_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_docreq_affected_24m, 0) end as friction_v3_docreq_affected_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_docreq_res_known_3m, 0) end as friction_v3_docreq_res_known_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_docreq_res_known_12m, 0) end as friction_v3_docreq_res_known_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_docreq_res_known_24m, 0) end as friction_v3_docreq_res_known_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_docreq_resolved_3m, 0) end as friction_v3_docreq_resolved_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_docreq_resolved_12m, 0) end as friction_v3_docreq_resolved_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.cov_friction_v3}') then coalesce(r.friction_v3_docreq_resolved_24m, 0) end as friction_v3_docreq_resolved_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.hist_start}') then coalesce(r.months_held_3m, 0) end as months_held_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.hist_start}') then coalesce(r.months_held_12m, 0) end as months_held_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.hist_start}') then coalesce(r.months_held_24m, 0) end as months_held_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.hist_start}') then coalesce(r.renewals_due_3m, 0) end as renewals_due_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.hist_start}') then coalesce(r.renewals_due_12m, 0) end as renewals_due_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.hist_start}') then coalesce(r.renewals_due_24m, 0) end as renewals_due_24m,
        case when date_add(add_months(r.MonthEndDate, -3), 1) >= to_date('${beh.hist_start}') then coalesce(r.motor_renewals_due_3m, 0) end as motor_renewals_due_3m,
        case when date_add(add_months(r.MonthEndDate, -12), 1) >= to_date('${beh.hist_start}') then coalesce(r.motor_renewals_due_12m, 0) end as motor_renewals_due_12m,
        case when date_add(add_months(r.MonthEndDate, -24), 1) >= to_date('${beh.hist_start}') then coalesce(r.motor_renewals_due_24m, 0) end as motor_renewals_due_24m
from    ${beh.stg}.beh_rolling r


## beh_portal_state


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_portal_state                                                              */
/*  Description : Per customer month-end with a failed renewal-portal log-in in the last 24         */
/*                months: the last failure in the window and the last successful log-in up to the   */
/*                month-end (curation #52).                                                         */
/*  Reads       : stg.beh_portal_month, beh.CustomerMonth                                           */
/*  Key         : SCV_Customer_Key + MonthEndDate                                                   */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_portal_state as
select  c.SCV_Customer_Key,
        c.MonthEndDate,
        s.LastFail24,
        s.LastSuccessToDate
from    ${beh.target}.CustomerMonth c
join    (select  SCV_Customer_Key,
                 MonthNo,
                 max(LastFailTs) over (partition by SCV_Customer_Key order by MonthNo
                                       range between 23 preceding and current row)          as LastFail24,
                 max(LastSuccessTs) over (partition by SCV_Customer_Key order by MonthNo
                                          range between unbounded preceding and current row) as LastSuccessToDate
         from    (select  SCV_Customer_Key, MonthNo, max(LastFailTs) as LastFailTs, max(LastSuccessTs) as LastSuccessTs
                  from    (select SCV_Customer_Key, MonthNo, LastFailTs, LastSuccessTs from ${beh.stg}.beh_portal_month
                           union all
                           select SCV_Customer_Key, MonthNo, cast(null as timestamp), cast(null as timestamp)
                           from   ${beh.target}.CustomerMonth) u
                  group by SCV_Customer_Key, MonthNo) g) s
        on  s.SCV_Customer_Key = c.SCV_Customer_Key
        and s.MonthNo = c.MonthNo
where   s.LastFail24 is not null


## beh_mfq_excess


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_mfq_excess                                                                */
/*  Description : Per customer month-end, the voluntary excess on the linked MFQ quotes of the      */
/*                last 24 months: count, range and most common value (curation #12).                */
/*  Reads       : stg.beh_mfq_quote_x, beh.CustomerMonth                                            */
/*  Key         : SCV_Customer_Key + MonthEndDate                                                   */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_mfq_excess as
select  c.SCV_Customer_Key,
        c.MonthEndDate,
        count(q.Excess)         as ExcessQuotes,
        max(q.Excess)           as MaxExcess,
        min(q.Excess)           as MinExcess,
        mode(q.Excess)          as ModalExcess
from    ${beh.target}.CustomerMonth c
join    ${beh.stg}.beh_mfq_quote_x q
        on  q.SCV_Customer_Key = c.SCV_Customer_Key
        and q.ActivityDate > add_months(c.MonthEndDate, -24)
        and q.ActivityDate <= c.MonthEndDate
where   q.Excess is not null
group by c.SCV_Customer_Key, c.MonthEndDate


## CustomerMonthBehaviour


In [ ]:
/****************************************************************************************************/
/*  Name        : beh.CustomerMonthBehaviour                                                        */
/*  Description : Rolling behaviour readings per customer month-end: the windowed counts,           */
/*                opportunity flags, per-month rates, rates per opportunity and recent-against-     */
/*                prior comparisons.                                                                */
/*  Reads       : stg.beh_behaviour_base, stg.beh_portal_state, stg.beh_mfq_excess                  */
/*  Grain       : customer x month-end (as beh.CustomerMonth)                                       */
/*  Key         : SCV_Customer_Key + MonthEndDate                                                   */
/****************************************************************************************************/
create or replace table ${beh.target}.CustomerMonthBehaviour as
select  b.*,
        b.calls_in_3m / nullif(b.months_held_3m, 0) as calls_in_pm_3m,
        b.calls_in_12m / nullif(b.months_held_12m, 0) as calls_in_pm_12m,
        b.calls_in_24m / nullif(b.months_held_24m, 0) as calls_in_pm_24m,
        b.calls_in_3m / nullif(b.months_held_3m, 0) - (b.calls_in_24m - b.calls_in_3m) / nullif(b.months_held_24m - b.months_held_3m, 0) as calls_in_recent_vs_prior,
        b.calls_in_il_3m / nullif(b.months_held_3m, 0) as calls_in_il_pm_3m,
        b.calls_in_il_12m / nullif(b.months_held_12m, 0) as calls_in_il_pm_12m,
        b.calls_in_il_24m / nullif(b.months_held_24m, 0) as calls_in_il_pm_24m,
        b.calls_in_il_3m / nullif(b.months_held_3m, 0) - (b.calls_in_il_24m - b.calls_in_il_3m) / nullif(b.months_held_24m - b.months_held_3m, 0) as calls_in_il_recent_vs_prior,
        b.inlife_contacts_3m / nullif(b.months_held_3m, 0) as inlife_contacts_pm_3m,
        b.inlife_contacts_12m / nullif(b.months_held_12m, 0) as inlife_contacts_pm_12m,
        b.inlife_contacts_24m / nullif(b.months_held_24m, 0) as inlife_contacts_pm_24m,
        b.inlife_contacts_3m / nullif(b.months_held_3m, 0) - (b.inlife_contacts_24m - b.inlife_contacts_3m) / nullif(b.months_held_24m - b.months_held_3m, 0) as inlife_contacts_recent_vs_prior,
        b.friction_events_3m / nullif(b.months_held_3m, 0) as friction_events_pm_3m,
        b.friction_events_12m / nullif(b.months_held_12m, 0) as friction_events_pm_12m,
        b.friction_events_24m / nullif(b.months_held_24m, 0) as friction_events_pm_24m,
        b.friction_events_3m / nullif(b.months_held_3m, 0) - (b.friction_events_24m - b.friction_events_3m) / nullif(b.months_held_24m - b.months_held_3m, 0) as friction_events_recent_vs_prior,
        b.friction_events_il_3m / nullif(b.months_held_3m, 0) as friction_events_il_pm_3m,
        b.friction_events_il_12m / nullif(b.months_held_12m, 0) as friction_events_il_pm_12m,
        b.friction_events_il_24m / nullif(b.months_held_24m, 0) as friction_events_il_pm_24m,
        b.friction_events_il_3m / nullif(b.months_held_3m, 0) - (b.friction_events_il_24m - b.friction_events_il_3m) / nullif(b.months_held_24m - b.months_held_3m, 0) as friction_events_il_recent_vs_prior,
        b.mtas_3m / nullif(b.months_held_3m, 0) as mtas_pm_3m,
        b.mtas_12m / nullif(b.months_held_12m, 0) as mtas_pm_12m,
        b.mtas_24m / nullif(b.months_held_24m, 0) as mtas_pm_24m,
        b.mtas_3m / nullif(b.months_held_3m, 0) - (b.mtas_24m - b.mtas_3m) / nullif(b.months_held_24m - b.months_held_3m, 0) as mtas_recent_vs_prior,
        b.mtas_il_3m / nullif(b.months_held_3m, 0) as mtas_il_pm_3m,
        b.mtas_il_12m / nullif(b.months_held_12m, 0) as mtas_il_pm_12m,
        b.mtas_il_24m / nullif(b.months_held_24m, 0) as mtas_il_pm_24m,
        b.mtas_il_3m / nullif(b.months_held_3m, 0) - (b.mtas_il_24m - b.mtas_il_3m) / nullif(b.months_held_24m - b.months_held_3m, 0) as mtas_il_recent_vs_prior,
        b.doc_requests_3m / nullif(b.months_held_3m, 0) as doc_requests_pm_3m,
        b.doc_requests_12m / nullif(b.months_held_12m, 0) as doc_requests_pm_12m,
        b.doc_requests_24m / nullif(b.months_held_24m, 0) as doc_requests_pm_24m,
        b.doc_requests_3m / nullif(b.months_held_3m, 0) - (b.doc_requests_24m - b.doc_requests_3m) / nullif(b.months_held_24m - b.months_held_3m, 0) as doc_requests_recent_vs_prior,
        b.chases_3m / nullif(b.months_held_3m, 0) as chases_pm_3m,
        b.chases_12m / nullif(b.months_held_12m, 0) as chases_pm_12m,
        b.chases_24m / nullif(b.months_held_24m, 0) as chases_pm_24m,
        b.chases_3m / nullif(b.months_held_3m, 0) - (b.chases_24m - b.chases_3m) / nullif(b.months_held_24m - b.months_held_3m, 0) as chases_recent_vs_prior,
        b.chases_il_3m / nullif(b.months_held_3m, 0) as chases_il_pm_3m,
        b.chases_il_12m / nullif(b.months_held_12m, 0) as chases_il_pm_12m,
        b.chases_il_24m / nullif(b.months_held_24m, 0) as chases_il_pm_24m,
        b.chases_il_3m / nullif(b.months_held_3m, 0) - (b.chases_il_24m - b.chases_il_3m) / nullif(b.months_held_24m - b.months_held_3m, 0) as chases_il_recent_vs_prior,
        b.failed_payments_3m / nullif(b.months_held_3m, 0) as failed_payments_pm_3m,
        b.failed_payments_12m / nullif(b.months_held_12m, 0) as failed_payments_pm_12m,
        b.failed_payments_24m / nullif(b.months_held_24m, 0) as failed_payments_pm_24m,
        b.failed_payments_3m / nullif(b.months_held_3m, 0) - (b.failed_payments_24m - b.failed_payments_3m) / nullif(b.months_held_24m - b.months_held_3m, 0) as failed_payments_recent_vs_prior,
        b.sales_calls_3m / nullif(b.months_held_3m, 0) as sales_calls_pm_3m,
        b.sales_calls_12m / nullif(b.months_held_12m, 0) as sales_calls_pm_12m,
        b.sales_calls_24m / nullif(b.months_held_24m, 0) as sales_calls_pm_24m,
        b.sales_calls_3m / nullif(b.months_held_3m, 0) - (b.sales_calls_24m - b.sales_calls_3m) / nullif(b.months_held_24m - b.months_held_3m, 0) as sales_calls_recent_vs_prior,
        b.sales_calls_il_3m / nullif(b.months_held_3m, 0) as sales_calls_il_pm_3m,
        b.sales_calls_il_12m / nullif(b.months_held_12m, 0) as sales_calls_il_pm_12m,
        b.sales_calls_il_24m / nullif(b.months_held_24m, 0) as sales_calls_il_pm_24m,
        b.sales_calls_il_3m / nullif(b.months_held_3m, 0) - (b.sales_calls_il_24m - b.sales_calls_il_3m) / nullif(b.months_held_24m - b.months_held_3m, 0) as sales_calls_il_recent_vs_prior,
        b.online_quotes_motor_3m / nullif(b.months_held_3m, 0) as online_quotes_motor_pm_3m,
        b.online_quotes_motor_12m / nullif(b.months_held_12m, 0) as online_quotes_motor_pm_12m,
        b.online_quotes_motor_24m / nullif(b.months_held_24m, 0) as online_quotes_motor_pm_24m,
        b.online_quotes_motor_3m / nullif(b.months_held_3m, 0) - (b.online_quotes_motor_24m - b.online_quotes_motor_3m) / nullif(b.months_held_24m - b.months_held_3m, 0) as online_quotes_motor_recent_vs_prior,
        b.online_quotes_home_3m / nullif(b.months_held_3m, 0) as online_quotes_home_pm_3m,
        b.online_quotes_home_12m / nullif(b.months_held_12m, 0) as online_quotes_home_pm_12m,
        b.online_quotes_home_24m / nullif(b.months_held_24m, 0) as online_quotes_home_pm_24m,
        b.online_quotes_home_3m / nullif(b.months_held_3m, 0) - (b.online_quotes_home_24m - b.online_quotes_home_3m) / nullif(b.months_held_24m - b.months_held_3m, 0) as online_quotes_home_recent_vs_prior,
        case when b.renewals_due_3m > 0 then 1 else 0 end as had_renewal_3m,
        case when b.msgs_sent_3m > 0 then 1 when b.msgs_sent_3m = 0 then 0 end as had_comms_3m,
        b.prompts_3m / nullif(b.renewals_due_3m, 0) as prompts_per_renewal_3m,
        b.ren_cust_actions_3m / nullif(b.renewals_due_3m, 0) as ren_actions_per_renewal_3m,
        b.concessions_3m / nullif(b.motor_renewals_due_3m, 0) as concessions_per_motor_renewal_3m,
        b.renewal_requotes_3m / nullif(b.renewals_due_3m, 0) as requotes_per_renewal_3m,
        b.msgs_opened_3m / nullif(b.msgs_sent_3m, 0) as open_rate_3m,
        b.msgs_clicked_3m / nullif(b.msgs_sent_3m, 0) as click_rate_3m,
        b.msgs_bounced_3m / nullif(b.msgs_sent_3m, 0) as bounce_rate_3m,
        case when b.mta_records_3m is not null then case when b.mta_cat_cover_3m > 0 then 1 else 0 end + case when b.mta_cat_drivers_3m > 0 then 1 else 0 end + case when b.mta_cat_vehicle_3m > 0 then 1 else 0 end + case when b.mta_cat_address_3m > 0 then 1 else 0 end + case when b.mta_cat_licence_3m > 0 then 1 else 0 end + case when b.mta_cat_use_3m > 0 then 1 else 0 end + case when b.mta_cat_personal_3m > 0 then 1 else 0 end end as mta_breadth_3m,
        b.mtas_elective_3m / nullif(b.mta_records_3m, 0) as mta_elective_share_3m,
        b.mtas_forced_3m / nullif(b.mta_records_3m, 0) as mta_forced_share_3m,
        b.episodes_friction_3m / nullif(b.episodes_3m, 0) as friction_episode_share_3m,
        b.episodes_completed_3m / nullif(b.episodes_3m, 0) as episode_completion_rate_3m,
        b.calls_in_abandoned_3m / nullif(b.calls_in_3m, 0) as calls_in_abandon_rate_3m,
        b.calls_in_long_wait_3m / nullif(b.calls_in_3m, 0) as calls_in_long_wait_rate_3m,
        b.calls_in_wait_s_3m / nullif(b.calls_in_3m, 0) as calls_in_avg_wait_s_3m,
        b.calls_in_talk_s_3m / nullif(b.calls_in_3m, 0) as calls_in_avg_talk_s_3m,
        b.msgs_opened_il_3m / nullif(b.msgs_sent_il_3m, 0) as open_rate_il_3m,
        b.msgs_clicked_il_3m / nullif(b.msgs_sent_il_3m, 0) as click_rate_il_3m,
        b.episodes_assisted_3m / nullif(b.episodes_3m, 0) as assisted_episode_share_3m,
        b.channel_switches_3m / nullif(b.episodes_channel_tagged_3m, 0) as channel_switches_per_episode_3m,
        b.episodes_repeat_contact_3m / nullif(b.episodes_3m, 0) as repeat_contact_episode_share_3m,
        b.nb_sales_web_assist_3m / nullif(b.nb_sales_3m, 0) as web_assist_rate_3m,
        b.doc_rejections_3m / nullif(b.doc_decisions_3m, 0) as doc_rejection_rate_3m,
        b.doc_attempts_3m / nullif(b.doc_decisions_3m, 0) as doc_attempts_per_doc_3m,
        b.mfq_retrievals_3m / nullif(b.mfq_quotes_3m, 0) as quote_retrieval_share_3m,
        b.mfq_insurers_3m / nullif(b.mfq_quotes_3m, 0) as insurers_per_quote_3m,
        b.mfq_returned_3m / nullif(b.mfq_quotes_3m, 0) as quote_return_share_3m,
        b.mfq_cover_down_3m / nullif(b.mfq_cover_compared_3m, 0) as cover_downgrade_share_3m,
        (b.mfq_cover_up_3m - b.mfq_cover_down_3m) / nullif(b.mfq_cover_compared_3m, 0) as cover_drift_net_3m,
        case when b.mfq_device_tagged_3m > 0 then case when b.mfq_device_switch_3m = 0 then 1 else 0 end end as device_consistent_3m,
        b.web_content_views_3m / nullif(b.web_page_views_3m, 0) as content_page_share_3m,
        case when b.web_sessions_3m > 0 then case when b.web_other_product_sessions_3m > 0 then 1 else 0 end end as other_product_viewed_3m,
        case when b.friction_all_3m > 0 then least(1.0, b.friction_resolved_3m / b.friction_all_3m) end as friction_resolved_share_3m,
        b.offers_clicked_3m / nullif(b.offers_sent_3m, 0) as offer_click_rate_3m,
        b.offer_clicks_nb_3m / nullif(b.offers_sent_3m, 0) as offer_response_nb_3m,
        b.offer_clicks_renewal_3m / nullif(b.offers_sent_3m, 0) as offer_response_renewal_3m,
        case when b.consent_withdrawals_3m > 0 then 1 when b.consent_withdrawals_3m = 0 then 0 end as consent_withdrawn_3m,
        case when b.friction_payment_3m > 0 then least(1.0, b.friction_payment_resolved_3m / b.friction_payment_3m) end as friction_payment_resolved_share_3m,
        case when b.friction_login_3m > 0 then least(1.0, b.friction_login_resolved_3m / b.friction_login_3m) end as friction_login_resolved_share_3m,
        case when b.friction_contact_3m > 0 then least(1.0, b.friction_contact_resolved_3m / b.friction_contact_3m) end as friction_contact_resolved_share_3m,
        case when b.friction_documents_3m > 0 then least(1.0, b.friction_documents_resolved_3m / b.friction_documents_3m) end as friction_documents_resolved_share_3m,
        case when b.friction_quote_3m > 0 then least(1.0, b.friction_quote_resolved_3m / b.friction_quote_3m) end as friction_quote_resolved_share_3m,
        case when b.friction_process_3m > 0 then least(1.0, b.friction_process_resolved_3m / b.friction_process_3m) end as friction_process_resolved_share_3m,
        case when b.friction_other_3m > 0 then least(1.0, b.friction_other_resolved_3m / b.friction_other_3m) end as friction_other_resolved_share_3m,
        b.friction_v3_affected_3m / nullif(b.friction_v3_eligible_3m, 0) as friction_v3_rate_3m,
        b.friction_v3_resolved_3m / nullif(b.friction_v3_res_known_3m, 0) as friction_v3_resolved_share_3m,
        b.friction_v3_renewal_affected_3m / nullif(b.friction_v3_renewal_eligible_3m, 0) as friction_v3_renewal_rate_3m,
        b.friction_v3_renewal_resolved_3m / nullif(b.friction_v3_renewal_res_known_3m, 0) as friction_v3_renewal_resolved_share_3m,
        b.friction_v3_firstyear_affected_3m / nullif(b.friction_v3_firstyear_eligible_3m, 0) as friction_v3_firstyear_rate_3m,
        b.friction_v3_firstyear_resolved_3m / nullif(b.friction_v3_firstyear_res_known_3m, 0) as friction_v3_firstyear_resolved_share_3m,
        b.friction_v3_change_affected_3m / nullif(b.friction_v3_change_eligible_3m, 0) as friction_v3_change_rate_3m,
        b.friction_v3_change_resolved_3m / nullif(b.friction_v3_change_res_known_3m, 0) as friction_v3_change_resolved_share_3m,
        b.friction_v3_docreq_affected_3m / nullif(b.friction_v3_docreq_eligible_3m, 0) as friction_v3_docreq_rate_3m,
        b.friction_v3_docreq_resolved_3m / nullif(b.friction_v3_docreq_res_known_3m, 0) as friction_v3_docreq_resolved_share_3m,
        case when b.renewals_due_12m > 0 then 1 else 0 end as had_renewal_12m,
        case when b.msgs_sent_12m > 0 then 1 when b.msgs_sent_12m = 0 then 0 end as had_comms_12m,
        b.prompts_12m / nullif(b.renewals_due_12m, 0) as prompts_per_renewal_12m,
        b.ren_cust_actions_12m / nullif(b.renewals_due_12m, 0) as ren_actions_per_renewal_12m,
        b.concessions_12m / nullif(b.motor_renewals_due_12m, 0) as concessions_per_motor_renewal_12m,
        b.renewal_requotes_12m / nullif(b.renewals_due_12m, 0) as requotes_per_renewal_12m,
        b.msgs_opened_12m / nullif(b.msgs_sent_12m, 0) as open_rate_12m,
        b.msgs_clicked_12m / nullif(b.msgs_sent_12m, 0) as click_rate_12m,
        b.msgs_bounced_12m / nullif(b.msgs_sent_12m, 0) as bounce_rate_12m,
        case when b.mta_records_12m is not null then case when b.mta_cat_cover_12m > 0 then 1 else 0 end + case when b.mta_cat_drivers_12m > 0 then 1 else 0 end + case when b.mta_cat_vehicle_12m > 0 then 1 else 0 end + case when b.mta_cat_address_12m > 0 then 1 else 0 end + case when b.mta_cat_licence_12m > 0 then 1 else 0 end + case when b.mta_cat_use_12m > 0 then 1 else 0 end + case when b.mta_cat_personal_12m > 0 then 1 else 0 end end as mta_breadth_12m,
        b.mtas_elective_12m / nullif(b.mta_records_12m, 0) as mta_elective_share_12m,
        b.mtas_forced_12m / nullif(b.mta_records_12m, 0) as mta_forced_share_12m,
        b.episodes_friction_12m / nullif(b.episodes_12m, 0) as friction_episode_share_12m,
        b.episodes_completed_12m / nullif(b.episodes_12m, 0) as episode_completion_rate_12m,
        b.calls_in_abandoned_12m / nullif(b.calls_in_12m, 0) as calls_in_abandon_rate_12m,
        b.calls_in_long_wait_12m / nullif(b.calls_in_12m, 0) as calls_in_long_wait_rate_12m,
        b.calls_in_wait_s_12m / nullif(b.calls_in_12m, 0) as calls_in_avg_wait_s_12m,
        b.calls_in_talk_s_12m / nullif(b.calls_in_12m, 0) as calls_in_avg_talk_s_12m,
        b.msgs_opened_il_12m / nullif(b.msgs_sent_il_12m, 0) as open_rate_il_12m,
        b.msgs_clicked_il_12m / nullif(b.msgs_sent_il_12m, 0) as click_rate_il_12m,
        b.episodes_assisted_12m / nullif(b.episodes_12m, 0) as assisted_episode_share_12m,
        b.channel_switches_12m / nullif(b.episodes_channel_tagged_12m, 0) as channel_switches_per_episode_12m,
        b.episodes_repeat_contact_12m / nullif(b.episodes_12m, 0) as repeat_contact_episode_share_12m,
        b.nb_sales_web_assist_12m / nullif(b.nb_sales_12m, 0) as web_assist_rate_12m,
        b.doc_rejections_12m / nullif(b.doc_decisions_12m, 0) as doc_rejection_rate_12m,
        b.doc_attempts_12m / nullif(b.doc_decisions_12m, 0) as doc_attempts_per_doc_12m,
        b.mfq_retrievals_12m / nullif(b.mfq_quotes_12m, 0) as quote_retrieval_share_12m,
        b.mfq_insurers_12m / nullif(b.mfq_quotes_12m, 0) as insurers_per_quote_12m,
        b.mfq_returned_12m / nullif(b.mfq_quotes_12m, 0) as quote_return_share_12m,
        b.mfq_cover_down_12m / nullif(b.mfq_cover_compared_12m, 0) as cover_downgrade_share_12m,
        (b.mfq_cover_up_12m - b.mfq_cover_down_12m) / nullif(b.mfq_cover_compared_12m, 0) as cover_drift_net_12m,
        case when b.mfq_device_tagged_12m > 0 then case when b.mfq_device_switch_12m = 0 then 1 else 0 end end as device_consistent_12m,
        b.web_content_views_12m / nullif(b.web_page_views_12m, 0) as content_page_share_12m,
        case when b.web_sessions_12m > 0 then case when b.web_other_product_sessions_12m > 0 then 1 else 0 end end as other_product_viewed_12m,
        case when b.friction_all_12m > 0 then least(1.0, b.friction_resolved_12m / b.friction_all_12m) end as friction_resolved_share_12m,
        b.offers_clicked_12m / nullif(b.offers_sent_12m, 0) as offer_click_rate_12m,
        b.offer_clicks_nb_12m / nullif(b.offers_sent_12m, 0) as offer_response_nb_12m,
        b.offer_clicks_renewal_12m / nullif(b.offers_sent_12m, 0) as offer_response_renewal_12m,
        case when b.consent_withdrawals_12m > 0 then 1 when b.consent_withdrawals_12m = 0 then 0 end as consent_withdrawn_12m,
        case when b.friction_payment_12m > 0 then least(1.0, b.friction_payment_resolved_12m / b.friction_payment_12m) end as friction_payment_resolved_share_12m,
        case when b.friction_login_12m > 0 then least(1.0, b.friction_login_resolved_12m / b.friction_login_12m) end as friction_login_resolved_share_12m,
        case when b.friction_contact_12m > 0 then least(1.0, b.friction_contact_resolved_12m / b.friction_contact_12m) end as friction_contact_resolved_share_12m,
        case when b.friction_documents_12m > 0 then least(1.0, b.friction_documents_resolved_12m / b.friction_documents_12m) end as friction_documents_resolved_share_12m,
        case when b.friction_quote_12m > 0 then least(1.0, b.friction_quote_resolved_12m / b.friction_quote_12m) end as friction_quote_resolved_share_12m,
        case when b.friction_process_12m > 0 then least(1.0, b.friction_process_resolved_12m / b.friction_process_12m) end as friction_process_resolved_share_12m,
        case when b.friction_other_12m > 0 then least(1.0, b.friction_other_resolved_12m / b.friction_other_12m) end as friction_other_resolved_share_12m,
        b.friction_v3_affected_12m / nullif(b.friction_v3_eligible_12m, 0) as friction_v3_rate_12m,
        b.friction_v3_resolved_12m / nullif(b.friction_v3_res_known_12m, 0) as friction_v3_resolved_share_12m,
        b.friction_v3_renewal_affected_12m / nullif(b.friction_v3_renewal_eligible_12m, 0) as friction_v3_renewal_rate_12m,
        b.friction_v3_renewal_resolved_12m / nullif(b.friction_v3_renewal_res_known_12m, 0) as friction_v3_renewal_resolved_share_12m,
        b.friction_v3_firstyear_affected_12m / nullif(b.friction_v3_firstyear_eligible_12m, 0) as friction_v3_firstyear_rate_12m,
        b.friction_v3_firstyear_resolved_12m / nullif(b.friction_v3_firstyear_res_known_12m, 0) as friction_v3_firstyear_resolved_share_12m,
        b.friction_v3_change_affected_12m / nullif(b.friction_v3_change_eligible_12m, 0) as friction_v3_change_rate_12m,
        b.friction_v3_change_resolved_12m / nullif(b.friction_v3_change_res_known_12m, 0) as friction_v3_change_resolved_share_12m,
        b.friction_v3_docreq_affected_12m / nullif(b.friction_v3_docreq_eligible_12m, 0) as friction_v3_docreq_rate_12m,
        b.friction_v3_docreq_resolved_12m / nullif(b.friction_v3_docreq_res_known_12m, 0) as friction_v3_docreq_resolved_share_12m,
        case when b.renewals_due_24m > 0 then 1 else 0 end as had_renewal_24m,
        case when b.msgs_sent_24m > 0 then 1 when b.msgs_sent_24m = 0 then 0 end as had_comms_24m,
        b.prompts_24m / nullif(b.renewals_due_24m, 0) as prompts_per_renewal_24m,
        b.ren_cust_actions_24m / nullif(b.renewals_due_24m, 0) as ren_actions_per_renewal_24m,
        b.concessions_24m / nullif(b.motor_renewals_due_24m, 0) as concessions_per_motor_renewal_24m,
        b.renewal_requotes_24m / nullif(b.renewals_due_24m, 0) as requotes_per_renewal_24m,
        b.msgs_opened_24m / nullif(b.msgs_sent_24m, 0) as open_rate_24m,
        b.msgs_clicked_24m / nullif(b.msgs_sent_24m, 0) as click_rate_24m,
        b.msgs_bounced_24m / nullif(b.msgs_sent_24m, 0) as bounce_rate_24m,
        case when b.mta_records_24m is not null then case when b.mta_cat_cover_24m > 0 then 1 else 0 end + case when b.mta_cat_drivers_24m > 0 then 1 else 0 end + case when b.mta_cat_vehicle_24m > 0 then 1 else 0 end + case when b.mta_cat_address_24m > 0 then 1 else 0 end + case when b.mta_cat_licence_24m > 0 then 1 else 0 end + case when b.mta_cat_use_24m > 0 then 1 else 0 end + case when b.mta_cat_personal_24m > 0 then 1 else 0 end end as mta_breadth_24m,
        b.mtas_elective_24m / nullif(b.mta_records_24m, 0) as mta_elective_share_24m,
        b.mtas_forced_24m / nullif(b.mta_records_24m, 0) as mta_forced_share_24m,
        b.episodes_friction_24m / nullif(b.episodes_24m, 0) as friction_episode_share_24m,
        b.episodes_completed_24m / nullif(b.episodes_24m, 0) as episode_completion_rate_24m,
        b.calls_in_abandoned_24m / nullif(b.calls_in_24m, 0) as calls_in_abandon_rate_24m,
        b.calls_in_long_wait_24m / nullif(b.calls_in_24m, 0) as calls_in_long_wait_rate_24m,
        b.calls_in_wait_s_24m / nullif(b.calls_in_24m, 0) as calls_in_avg_wait_s_24m,
        b.calls_in_talk_s_24m / nullif(b.calls_in_24m, 0) as calls_in_avg_talk_s_24m,
        b.msgs_opened_il_24m / nullif(b.msgs_sent_il_24m, 0) as open_rate_il_24m,
        b.msgs_clicked_il_24m / nullif(b.msgs_sent_il_24m, 0) as click_rate_il_24m,
        b.episodes_assisted_24m / nullif(b.episodes_24m, 0) as assisted_episode_share_24m,
        b.channel_switches_24m / nullif(b.episodes_channel_tagged_24m, 0) as channel_switches_per_episode_24m,
        b.episodes_repeat_contact_24m / nullif(b.episodes_24m, 0) as repeat_contact_episode_share_24m,
        b.nb_sales_web_assist_24m / nullif(b.nb_sales_24m, 0) as web_assist_rate_24m,
        b.doc_rejections_24m / nullif(b.doc_decisions_24m, 0) as doc_rejection_rate_24m,
        b.doc_attempts_24m / nullif(b.doc_decisions_24m, 0) as doc_attempts_per_doc_24m,
        b.mfq_retrievals_24m / nullif(b.mfq_quotes_24m, 0) as quote_retrieval_share_24m,
        b.mfq_insurers_24m / nullif(b.mfq_quotes_24m, 0) as insurers_per_quote_24m,
        b.mfq_returned_24m / nullif(b.mfq_quotes_24m, 0) as quote_return_share_24m,
        b.mfq_cover_down_24m / nullif(b.mfq_cover_compared_24m, 0) as cover_downgrade_share_24m,
        (b.mfq_cover_up_24m - b.mfq_cover_down_24m) / nullif(b.mfq_cover_compared_24m, 0) as cover_drift_net_24m,
        case when b.mfq_device_tagged_24m > 0 then case when b.mfq_device_switch_24m = 0 then 1 else 0 end end as device_consistent_24m,
        b.web_content_views_24m / nullif(b.web_page_views_24m, 0) as content_page_share_24m,
        case when b.web_sessions_24m > 0 then case when b.web_other_product_sessions_24m > 0 then 1 else 0 end end as other_product_viewed_24m,
        case when b.friction_all_24m > 0 then least(1.0, b.friction_resolved_24m / b.friction_all_24m) end as friction_resolved_share_24m,
        b.offers_clicked_24m / nullif(b.offers_sent_24m, 0) as offer_click_rate_24m,
        b.offer_clicks_nb_24m / nullif(b.offers_sent_24m, 0) as offer_response_nb_24m,
        b.offer_clicks_renewal_24m / nullif(b.offers_sent_24m, 0) as offer_response_renewal_24m,
        case when b.consent_withdrawals_24m > 0 then 1 when b.consent_withdrawals_24m = 0 then 0 end as consent_withdrawn_24m,
        case when b.friction_payment_24m > 0 then least(1.0, b.friction_payment_resolved_24m / b.friction_payment_24m) end as friction_payment_resolved_share_24m,
        case when b.friction_login_24m > 0 then least(1.0, b.friction_login_resolved_24m / b.friction_login_24m) end as friction_login_resolved_share_24m,
        case when b.friction_contact_24m > 0 then least(1.0, b.friction_contact_resolved_24m / b.friction_contact_24m) end as friction_contact_resolved_share_24m,
        case when b.friction_documents_24m > 0 then least(1.0, b.friction_documents_resolved_24m / b.friction_documents_24m) end as friction_documents_resolved_share_24m,
        case when b.friction_quote_24m > 0 then least(1.0, b.friction_quote_resolved_24m / b.friction_quote_24m) end as friction_quote_resolved_share_24m,
        case when b.friction_process_24m > 0 then least(1.0, b.friction_process_resolved_24m / b.friction_process_24m) end as friction_process_resolved_share_24m,
        case when b.friction_other_24m > 0 then least(1.0, b.friction_other_resolved_24m / b.friction_other_24m) end as friction_other_resolved_share_24m,
        b.friction_v3_affected_24m / nullif(b.friction_v3_eligible_24m, 0) as friction_v3_rate_24m,
        b.friction_v3_resolved_24m / nullif(b.friction_v3_res_known_24m, 0) as friction_v3_resolved_share_24m,
        b.friction_v3_renewal_affected_24m / nullif(b.friction_v3_renewal_eligible_24m, 0) as friction_v3_renewal_rate_24m,
        b.friction_v3_renewal_resolved_24m / nullif(b.friction_v3_renewal_res_known_24m, 0) as friction_v3_renewal_resolved_share_24m,
        b.friction_v3_firstyear_affected_24m / nullif(b.friction_v3_firstyear_eligible_24m, 0) as friction_v3_firstyear_rate_24m,
        b.friction_v3_firstyear_resolved_24m / nullif(b.friction_v3_firstyear_res_known_24m, 0) as friction_v3_firstyear_resolved_share_24m,
        b.friction_v3_change_affected_24m / nullif(b.friction_v3_change_eligible_24m, 0) as friction_v3_change_rate_24m,
        b.friction_v3_change_resolved_24m / nullif(b.friction_v3_change_res_known_24m, 0) as friction_v3_change_resolved_share_24m,
        b.friction_v3_docreq_affected_24m / nullif(b.friction_v3_docreq_eligible_24m, 0) as friction_v3_docreq_rate_24m,
        b.friction_v3_docreq_resolved_24m / nullif(b.friction_v3_docreq_res_known_24m, 0) as friction_v3_docreq_resolved_share_24m,
        b.msgs_opened_3m / nullif(b.msgs_sent_3m, 0) - (b.msgs_opened_24m - b.msgs_opened_3m) / nullif(b.msgs_sent_24m - b.msgs_sent_3m, 0) as open_rate_recent_vs_prior,
        b.msgs_clicked_3m / nullif(b.msgs_sent_3m, 0) - (b.msgs_clicked_24m - b.msgs_clicked_3m) / nullif(b.msgs_sent_24m - b.msgs_sent_3m, 0) as click_rate_recent_vs_prior,
        case when x.LastFail24 is null then null when x.LastSuccessToDate > x.LastFail24 then 0 else 1 end as failed_login_unrecovered_24m,
        case when e.ExcessQuotes >= 2 then (e.MaxExcess - e.MinExcess) / nullif(e.ModalExcess, 0) end as excess_elasticity_24m
from    ${beh.stg}.beh_behaviour_base b
left join ${beh.stg}.beh_portal_state x
        on  x.SCV_Customer_Key = b.SCV_Customer_Key
        and x.MonthEndDate = b.MonthEndDate
left join ${beh.stg}.beh_mfq_excess e
        on  e.SCV_Customer_Key = b.SCV_Customer_Key
        and e.MonthEndDate = b.MonthEndDate


## Checks


In [ ]:
%%pyspark
# Row count, duplicate keys and null keys for every table this notebook built. Logged to BehBuildLog;
# the cell fails if any check failed, naming the table and the likely cause.
beh_check("BEH_30_Rolling", [{"table": "{stg}.beh_grid", "key": ["SCV_Customer_Key", "MonthNo"]}, {"table": "{stg}.beh_rolling", "key": ["SCV_Customer_Key", "MonthEndDate"]}, {"table": "{stg}.beh_behaviour_base", "key": ["SCV_Customer_Key", "MonthEndDate"]}, {"table": "{stg}.beh_portal_state", "key": ["SCV_Customer_Key", "MonthEndDate"]}, {"table": "{stg}.beh_mfq_excess", "key": ["SCV_Customer_Key", "MonthEndDate"]}, {"table": "{target}.CustomerMonthBehaviour", "key": ["SCV_Customer_Key", "MonthEndDate"]}])


In [ ]:
%%pyspark
beh_run_log("BEH_30_Rolling", "succeeded")
